# Biohub L2 K-DN: DaughterNet training with honest folds + OOF predictions

Trains `tools/daughternet.py` (DivNet UNet-GAP body, 3-logit head, classes cont / parent / daughter) **from random init**
on the 199 train videos with folds grouped by video, and predicts every labelled sample with the fold model that never
saw its video.

* Folds: `experiments/l2_daughternet/splits.json` (tools/make_splits.py, 5 folds) embedded verbatim.
* Labels (annotated GT nodes only): parent = >= 2 GT children; daughter = child of a parent at its first frame;
  cont = in/out-degree 1 and >= 2 frames (lineage) from any parent/daughter node, <= 60 per video; hard negatives =
  cont nodes inside the L2 insertion gate of an annotated one-child node (as u or v), <= 40 per video.
* Input = the DivNet contract (`prepare_sample`): lags -1..+2, XY mean-pool 4, 16x32x32 crop, percentile 50/99.5
  clipped to [-0.5, 6], gaussian marker -> (5, 16, 32, 32). An 18x34x34 crop is cached so the +-1 voxel jitter has a
  margin; its inner crop equals the contract sample exactly (asserted per video).
* Training: AdamW 1e-3 + cosine, <= 30 epochs, early stopping on the fold's val loss (class-macro CE), batch 64,
  class-balanced sampler, flips z/y/x + rot90 yx (all 5 channels), +-1 voxel jitter, gain 0.8-1.2. Two folds run
  concurrently, one per T4 (one subprocess per fold).
* Outputs in /kaggle/working: `oof.csv`, `oof_readout.json`, `folds/fold_k/{best,last}.pt`
  ({'model_state','config','fold','epoch','val_loss'}), `folds/fold_k/{history,summary}.json`,
  `labels_summary.csv`, `splits_used.json`, `contract_check.json`.
* Knobs (env, set in the config cell): KDN_SMOKE=1 (5 videos, 1 epoch, 2 folds), KDN_MAX_VIDEOS, KDN_EPOCHS,
  KDN_EPOCH_SAMPLES, KDN_PATIENCE, KDN_INNER_VAL, KDN_CACHE_DTYPE, ...


In [ ]:
import os, sys, platform, importlib
from pathlib import Path

KDN_SRC = Path("/kaggle/working/kdn_src")
(KDN_SRC / "tools").mkdir(parents=True, exist_ok=True)
print("python", sys.version.split()[0], platform.platform(), "cpus", os.cpu_count())
import torch
print("torch", torch.__version__, "cuda", torch.cuda.is_available(), torch.cuda.device_count(),
      [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
for _name in ("numpy", "pandas", "blosc2", "numcodecs", "zstandard", "pyarrow", "imagecodecs", "zarr"):
    try:
        _mod = importlib.import_module(_name)
        print(f"  {_name:12s} {getattr(_mod, '__version__', '?')}")
    except Exception as _exc:
        print(f"  {_name:12s} MISSING ({type(_exc).__name__})")
for _p in sorted(Path("/kaggle/input").glob("*")) if Path("/kaggle/input").exists() else []:
    print("input:", _p)


In [ ]:
%%writefile /kaggle/working/kdn_src/tools/__init__.py
# tools package materialised by the K-DN kernel (divnet_infer + daughternet, verbatim repo copies)


In [ ]:
%%writefile /kaggle/working/kdn_src/tools/divnet_infer.py
"""Inference for the public ``giorgosi/biohub-divnet-v2`` checkpoint.

The dataset manifest is the contract: four image channels at lags -1, 0, +1,
+2 plus a Gaussian centre marker, XY mean pooling by four, a 16x32x32 crop,
percentile normalisation, and a small 3-D U-Net followed by global pooling.

``read_test_frame`` is supplied by the caller (the public pipeline notebook or
the local B20 scorer). This module raises on an incompatible checkpoint or
tensor shape; a silent geometric fallback would invalidate B20.
"""

from __future__ import annotations

import os
from pathlib import Path
from typing import Callable

import numpy as np
import torch


class _DivNetConvBlock(torch.nn.Module):
    def __init__(self, in_channels: int, out_channels: int) -> None:
        super().__init__()
        self.block = torch.nn.Sequential(
            torch.nn.Conv3d(in_channels, out_channels, 3, padding=1, bias=False),
            torch.nn.GroupNorm(min(8, out_channels), out_channels),
            torch.nn.SiLU(inplace=True),
            torch.nn.Conv3d(out_channels, out_channels, 3, padding=1, bias=False),
            torch.nn.GroupNorm(min(8, out_channels), out_channels),
            torch.nn.SiLU(inplace=True),
        )

    def forward(self, value: torch.Tensor) -> torch.Tensor:
        return self.block(value)


class DivNetMitosisClassifier(torch.nn.Module):
    """Architecture reconstructed exactly from the public checkpoint shapes."""

    def __init__(self, in_channels: int = 5, base_channels: int = 16) -> None:
        super().__init__()
        c = int(base_channels)
        self.enc1 = _DivNetConvBlock(in_channels, c)
        self.pool1 = torch.nn.MaxPool3d(2, 2)
        self.enc2 = _DivNetConvBlock(c, c * 2)
        self.pool2 = torch.nn.MaxPool3d(2, 2)
        self.enc3 = _DivNetConvBlock(c * 2, c * 4)
        self.pool3 = torch.nn.MaxPool3d(2, 2)
        self.bottleneck = _DivNetConvBlock(c * 4, c * 8)
        self.up3 = torch.nn.ConvTranspose3d(c * 8, c * 4, 2, 2)
        self.dec3 = _DivNetConvBlock(c * 8, c * 4)
        self.up2 = torch.nn.ConvTranspose3d(c * 4, c * 2, 2, 2)
        self.dec2 = _DivNetConvBlock(c * 4, c * 2)
        self.up1 = torch.nn.ConvTranspose3d(c * 2, c, 2, 2)
        self.dec1 = _DivNetConvBlock(c * 2, c)
        self.gap = torch.nn.AdaptiveAvgPool3d((1, 1, 1))
        self.head = torch.nn.Linear(c, 1)

    def forward(self, value: torch.Tensor) -> torch.Tensor:
        e1 = self.enc1(value)
        e2 = self.enc2(self.pool1(e1))
        e3 = self.enc3(self.pool2(e2))
        bottleneck = self.bottleneck(self.pool3(e3))
        d3 = self.dec3(torch.cat([self.up3(bottleneck), e3], dim=1))
        d2 = self.dec2(torch.cat([self.up2(d3), e2], dim=1))
        d1 = self.dec1(torch.cat([self.up1(d2), e1], dim=1))
        return self.head(self.gap(d1).flatten(1))


def _checkpoint_candidates() -> list[Path]:
    root = Path(__file__).resolve().parent.parent
    explicit = os.environ.get("BIOHUB_DIVNET_CHECKPOINT", "").strip()
    candidates = [Path(explicit)] if explicit else []
    candidates.extend(
        [
            root / "external" / "giorgosi__biohub-divnet-v2" / "best_overall.pt",
            root / "external" / "giorgosi__biohub-divnet-v2" / "weights" / "divnet" / "best_overall.pt",
            Path("/kaggle/input/biohub-divnet-v2/best_overall.pt"),
            Path("/kaggle/input/biohub-divnet-v2/weights/divnet/best_overall.pt"),
            Path("/kaggle/input/datasets/giorgosi/biohub-divnet-v2/best_overall.pt"),
            Path("/kaggle/input/datasets/giorgosi/biohub-divnet-v2/weights/divnet/best_overall.pt"),
        ]
    )
    return candidates


def load_divnet_mitosis_model(checkpoint: Path | None = None) -> dict[str, object]:
    if not globals().get("DIVNET_VERIFY", True):
        raise RuntimeError("DivNet is disabled by configuration")
    path = Path(checkpoint) if checkpoint is not None else next(
        (item for item in _checkpoint_candidates() if item.is_file()), None
    )
    if path is None:
        checked = "\n".join(str(item) for item in _checkpoint_candidates())
        raise FileNotFoundError(f"DivNet checkpoint not found. Checked:\n{checked}")
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    payload = torch.load(path, map_location="cpu", weights_only=False)
    state = payload.get("model_state", payload) if isinstance(payload, dict) else payload
    config = payload.get("config", {}) if isinstance(payload, dict) else {}
    model_cfg = config.get("model", {}) if isinstance(config, dict) else {}
    model = DivNetMitosisClassifier(
        in_channels=int(model_cfg.get("in_channels", 5)),
        base_channels=int(model_cfg.get("base_channels", 16)),
    )
    model.load_state_dict(state, strict=True)
    model.to(device).eval()
    return {
        "model": model,
        "device": device,
        "torch": torch,
        "path": path.resolve(),
        "config": config,
        "prepared_frames": {},
    }


def _pool_xy(frame: np.ndarray, factor: int) -> np.ndarray:
    frame = np.asarray(frame)
    if factor <= 1:
        return frame.astype(np.float32, copy=False)
    z, y, x = frame.shape
    y2, x2 = (y // factor) * factor, (x // factor) * factor
    value = frame[:, :y2, :x2].astype(np.float32, copy=False)
    return value.reshape(z, y2 // factor, factor, x2 // factor, factor).mean(axis=(2, 4))


def _normalise(frame: np.ndarray, lo_pct: float, hi_pct: float, clip: tuple[float, float]) -> np.ndarray:
    value = np.asarray(frame, dtype=np.float32)
    lo, hi = np.percentile(value, [lo_pct, hi_pct])
    if not np.isfinite(lo) or not np.isfinite(hi) or hi <= lo:
        return np.zeros_like(value)
    return np.clip((value - float(lo)) / (float(hi) - float(lo)), clip[0], clip[1]).astype(np.float32)


def _crop_with_padding(frame: np.ndarray, centre: tuple[int, int, int], shape: tuple[int, int, int]) -> np.ndarray:
    result = np.zeros(shape, dtype=np.float32)
    source_slices: list[slice] = []
    target_slices: list[slice] = []
    for coordinate, size, limit in zip(centre, shape, frame.shape):
        start = coordinate - size // 2
        stop = start + size
        source_start, source_stop = max(0, start), min(limit, stop)
        target_start = source_start - start
        target_stop = target_start + max(0, source_stop - source_start)
        source_slices.append(slice(source_start, source_stop))
        target_slices.append(slice(target_start, target_stop))
    result[tuple(target_slices)] = frame[tuple(source_slices)]
    return result


def _marker(shape: tuple[int, int, int], sigma: tuple[float, float, float]) -> np.ndarray:
    axes = [np.arange(size, dtype=np.float32) - size // 2 for size in shape]
    zz, yy, xx = np.meshgrid(*axes, indexing="ij")
    distance = (zz / sigma[0]) ** 2 + (yy / sigma[1]) ** 2 + (xx / sigma[2]) ** 2
    return np.exp(-0.5 * distance).astype(np.float32)


def prepare_divnet_input(
    dataset: str,
    t: int,
    parent_node: dict[str, object],
    divnet_bundle: dict[str, object],
    frame_cache: dict[object, np.ndarray],
    read_frame: Callable[[str, int, dict], np.ndarray] | None = None,
) -> np.ndarray:
    """Build one ``(5, 16, 32, 32)`` sample from the manifest recipe."""
    cfg = divnet_bundle.get("config", {})
    input_cfg = cfg.get("input", {}) if isinstance(cfg, dict) else {}
    norm_cfg = cfg.get("normalization", {}) if isinstance(cfg, dict) else {}
    lags = tuple(int(value) for value in input_cfg.get("image_lags", (-1, 0, 1, 2)))
    pool_xy = int(input_cfg.get("pool_xy", 4))
    shape = (
        int(input_cfg.get("crop_z", 16)),
        int(input_cfg.get("crop_yx", 32)),
        int(input_cfg.get("crop_yx", 32)),
    )
    sigma = tuple(float(value) for value in input_cfg.get("marker_sigma", (1.5, 2.0, 2.0)))
    clip = tuple(float(value) for value in norm_cfg.get("clip", (-0.5, 6.0)))
    reader = read_frame or globals().get("read_test_frame")
    if reader is None:
        raise RuntimeError("DivNet needs a read_frame callback or a global read_test_frame")
    prepared = divnet_bundle.setdefault("prepared_frames", {})
    channels: list[np.ndarray] = []
    for lag in lags:
        frame_t = max(0, int(t) + lag)
        key = (dataset, frame_t)
        if key not in prepared:
            raw = reader(dataset, frame_t, frame_cache)
            pooled = _pool_xy(raw, pool_xy)
            prepared[key] = _normalise(
                pooled,
                float(norm_cfg.get("lo_pct", 50.0)),
                float(norm_cfg.get("hi_pct", 99.5)),
                clip,
            )
            frame_cache.pop(frame_t, None)
        centre = (
            int(round(float(parent_node["z"]))),
            int(round(float(parent_node["y"]) / pool_xy)),
            int(round(float(parent_node["x"]) / pool_xy)),
        )
        channels.append(_crop_with_padding(prepared[key], centre, shape))
    channels.append(_marker(shape, sigma))
    sample = np.stack(channels, axis=0)
    if sample.shape != (5, *shape) or not np.isfinite(sample).all():
        raise RuntimeError(f"Invalid DivNet input: shape={sample.shape}, finite={np.isfinite(sample).all()}")
    return sample


def divnet_score_division(
    dataset: str,
    t: int,
    parent_node: dict[str, object],
    daughter1: dict[str, object],
    daughter2: dict[str, object],
    divnet_bundle: dict[str, object],
    frame_cache: dict[object, np.ndarray],
    read_frame: Callable[[str, int, dict], np.ndarray] | None = None,
) -> float:
    """Return the parent division probability; daughter arguments preserve the public API."""
    del daughter1, daughter2
    sample = prepare_divnet_input(dataset, t, parent_node, divnet_bundle, frame_cache, read_frame)
    torch_mod = divnet_bundle["torch"]
    tensor = torch_mod.from_numpy(sample).unsqueeze(0).to(
        device=divnet_bundle["device"], dtype=torch_mod.float32
    )
    with torch_mod.inference_mode():
        logit = divnet_bundle["model"](tensor)
        probability = float(torch_mod.sigmoid(logit).item())
    if not np.isfinite(probability):
        raise RuntimeError("DivNet produced a non-finite probability")
    return probability


DIVNET_BUNDLE = None


In [ ]:
%%writefile /kaggle/working/kdn_src/tools/daughternet.py
"""DaughterNet: 3-class node classifier (continuation / parent / daughter) for the L2 division stage.

Shared contract between training (Kaggle K-DN kernels), the submission kernel patch (--patch divinsert) and the laptop
replay. The input sample is EXACTLY the DivNet manifest contract of tools/divnet_infer.prepare_divnet_input: image
channels at lags -1, 0, +1, +2 around the node's frame t, XY mean-pool by 4, a 16x32x32 crop centred on the node,
percentile 50/99.5 normalisation clipped to [-0.5, 6], plus one gaussian centre marker -> (5, 16, 32, 32).
The network is the DivNet UNet-GAP body with a 3-logit head. Class order is fixed: CLASSES = (cont, parent, daughter).
  parent   = GT dividing node at the last frame before the split (label at its own t)
  daughter = GT daughter node at its first frame (label at its own t)
  cont     = annotated node with in-degree = out-degree = 1, >= 2 frames from any GT division
Scoring averages softmax over the loaded fold models (ensemble) or uses one fold model per video (OOF).
"""
from __future__ import annotations

from pathlib import Path
from typing import Callable, Sequence

import numpy as np
import torch

try:  # repo layout
    from tools.divnet_infer import _DivNetConvBlock, prepare_divnet_input
except Exception:  # embedded in a notebook cell after divnet_infer's code (names already global)
    _DivNetConvBlock = globals().get("_DivNetConvBlock")
    prepare_divnet_input = globals().get("prepare_divnet_input")

CLASSES = ("cont", "parent", "daughter")
DEFAULT_CFG = {
    "input": {"image_lags": [-1, 0, 1, 2], "pool_xy": 4, "crop_z": 16, "crop_yx": 32, "marker_sigma": [1.5, 2.0, 2.0]},
    "normalization": {"lo_pct": 50.0, "hi_pct": 99.5, "clip": [-0.5, 6.0]},
    "model": {"in_channels": 5, "base_channels": 16, "n_classes": 3},
}


class DaughterNet(torch.nn.Module):
    """DivNet UNet-GAP body (same layer names as DivNetMitosisClassifier) with an n_classes head."""

    def __init__(self, in_channels: int = 5, base_channels: int = 16, n_classes: int = 3) -> None:
        super().__init__()
        c = int(base_channels)
        self.enc1 = _DivNetConvBlock(in_channels, c)
        self.pool1 = torch.nn.MaxPool3d(2, 2)
        self.enc2 = _DivNetConvBlock(c, c * 2)
        self.pool2 = torch.nn.MaxPool3d(2, 2)
        self.enc3 = _DivNetConvBlock(c * 2, c * 4)
        self.pool3 = torch.nn.MaxPool3d(2, 2)
        self.bottleneck = _DivNetConvBlock(c * 4, c * 8)
        self.up3 = torch.nn.ConvTranspose3d(c * 8, c * 4, 2, 2)
        self.dec3 = _DivNetConvBlock(c * 8, c * 4)
        self.up2 = torch.nn.ConvTranspose3d(c * 4, c * 2, 2, 2)
        self.dec2 = _DivNetConvBlock(c * 4, c * 2)
        self.up1 = torch.nn.ConvTranspose3d(c * 2, c, 2, 2)
        self.dec1 = _DivNetConvBlock(c * 2, c)
        self.gap = torch.nn.AdaptiveAvgPool3d((1, 1, 1))
        self.head = torch.nn.Linear(c, int(n_classes))

    def forward(self, value: torch.Tensor) -> torch.Tensor:
        e1 = self.enc1(value)
        e2 = self.enc2(self.pool1(e1))
        e3 = self.enc3(self.pool2(e2))
        b = self.bottleneck(self.pool3(e3))
        d3 = self.dec3(torch.cat([self.up3(b), e3], dim=1))
        d2 = self.dec2(torch.cat([self.up2(d3), e2], dim=1))
        d1 = self.dec1(torch.cat([self.up1(d2), e1], dim=1))
        return self.head(self.gap(d1).flatten(1))


def build_model(cfg: dict | None = None) -> DaughterNet:
    m = (cfg or DEFAULT_CFG)["model"]
    return DaughterNet(int(m["in_channels"]), int(m["base_channels"]), int(m["n_classes"]))


def load_daughternet(paths: Sequence[Path | str], device: str | None = None) -> dict:
    """Load one or more fold checkpoints ({'model_state', 'config'}) -> bundle usable by prepare_sample/score_nodes."""
    dev = torch.device(device or ("cuda" if torch.cuda.is_available() else "cpu"))
    models, cfg = [], None
    for p in paths:
        payload = torch.load(Path(p), map_location="cpu", weights_only=False)
        cfg = payload.get("config", DEFAULT_CFG) if isinstance(payload, dict) else DEFAULT_CFG
        model = build_model(cfg)
        model.load_state_dict(payload["model_state"] if "model_state" in payload else payload, strict=True)
        models.append(model.to(dev).eval())
    if not models:
        raise FileNotFoundError("no DaughterNet checkpoints given")
    return {"models": models, "device": dev, "torch": torch, "config": cfg, "prepared_frames": {}}


def prepare_sample(dataset: str, t: int, node: dict, bundle: dict, frame_cache: dict,
                   read_frame: Callable[[str, int, dict], np.ndarray] | None = None) -> np.ndarray:
    """(5, 16, 32, 32) float32 sample, identical to the DivNet contract (bundle['config'] carries input/normalization)."""
    return prepare_divnet_input(dataset, int(t), node, bundle, frame_cache, read_frame)


@torch.no_grad()
def score_nodes(dataset: str, nodes: Sequence[dict], bundle: dict, frame_cache: dict,
                read_frame: Callable[[str, int, dict], np.ndarray] | None = None, batch: int = 64,
                model_index: int | None = None) -> np.ndarray:
    """Softmax probabilities (N, 3) in CLASSES order; mean over fold models unless model_index selects one (OOF)."""
    if not nodes:
        return np.zeros((0, 3), dtype=np.float32)
    order = sorted(range(len(nodes)), key=lambda i: int(nodes[i]["t"]))
    out = np.zeros((len(nodes), 3), dtype=np.float32)
    models = bundle["models"] if model_index is None else [bundle["models"][model_index]]
    dev = bundle["device"]
    for s in range(0, len(order), batch):
        idx = order[s:s + batch]
        x = np.stack([prepare_sample(dataset, int(nodes[i]["t"]), nodes[i], bundle, frame_cache, read_frame) for i in idx])
        xt = torch.from_numpy(x).to(dev, dtype=torch.float32)
        p = torch.stack([torch.softmax(m(xt), dim=1) for m in models]).mean(0)
        out[idx] = p.float().cpu().numpy()
    return out


In [ ]:
%%writefile /kaggle/working/kdn_src/kdn_core.py
"""K-DN: train DaughterNet (tools/daughternet.py) from scratch with honest video folds and write out-of-fold predictions.

One module shared by the Kaggle kernel (kdn-train.ipynb materialises it with %%writefile next to tools/divnet_infer.py
and tools/daughternet.py) and by the laptop runner run_local.py. Steps (run_all runs them in order):

  step_prepare      locate the data, folds (embedded splits.json, else the inline seed-0 rule), GT labels per video
  step_build_cache  DivNet-contract crops for every labelled node -> one memmap (N, 4, 18, 34, 34) + meta.csv
  step_train        one subprocess per fold (two at once on 2 GPUs): balanced sampler, flips/rot90/jitter/gain,
                    AdamW + cosine, early stopping on the fold's val loss, best.pt/last.pt every epoch, fold OOF
  step_oof          oof.csv + oof_readout.json (+ in-sample DivNet comparison)

Labels (annotated GT nodes only, never unannotated space):
  parent   node with >= 2 GT children (at its own t)
  daughter each child of a parent (at its own, first, frame)
  cont     in-degree 1, out-degree 1, >= cont_min_dist frames (lineage) from every parent AND daughter node,
           random subsample of cont_per_video per video
  hardneg  cont nodes that sit in the L2 insertion gate of an annotated one-child GT node, as u or as v
           (u: exactly one child c1 at t+1 within 10 um; v at t+1, v != c1, within 15 um of u; 3 <= |c1-v| <= 20 um;
           |midpoint(c1, v) - u| <= 6.5 um; cos(c1-u, v-u) <= 0.25), up to hardneg_per_video, label cont
Crops: prepare_sample (DivNet contract) called with an enlarged 18x34x34 crop; the inner 16x32x32 of it equals the
contract sample exactly (asserted on the first nodes of every video), the 1-voxel margin feeds the +-1 voxel jitter.
Node coordinates are voxel indices of the full-resolution (64, 256, 256) frame, exactly as in the GT .geff.
Frame index t+lag is clamped to [0, T-1] (the contract clamps the low end with max(0, .); the high end is clamped here,
as in friend A's FrameReader, so nodes at t >= T-2 get a repeated last frame).
"""
from __future__ import annotations

import copy
import json
import math
import os
import struct
import subprocess
import sys
import threading
import time
import zlib
from pathlib import Path

import numpy as np

try:  # repo layout (laptop) and the Kaggle kernel (tools/ materialised under kdn_src)
    from tools.daughternet import CLASSES, DEFAULT_CFG, build_model, load_daughternet, prepare_sample, score_nodes
    from tools.divnet_infer import _marker, load_divnet_mitosis_model, _checkpoint_candidates
except Exception:  # pragma: no cover - notebook fallback when the cells were executed into globals
    CLASSES = globals()["CLASSES"]
    DEFAULT_CFG = globals()["DEFAULT_CFG"]
    build_model = globals()["build_model"]
    load_daughternet = globals()["load_daughternet"]
    prepare_sample = globals()["prepare_sample"]
    score_nodes = globals()["score_nodes"]
    _marker = globals()["_marker"]
    load_divnet_mitosis_model = globals()["load_divnet_mitosis_model"]
    _checkpoint_candidates = globals().get("_checkpoint_candidates", lambda: [])

COMPETITION = "biohub-cell-tracking-during-development"
VOXEL_SCALE_UM = np.array([1.625, 0.40625, 0.40625], dtype=np.float64)
CLASS_INDEX = {name: i for i, name in enumerate(CLASSES)}
CROP = (16, 32, 32)          # DivNet contract crop (z, y, x) in pooled voxels
MARGIN = 1                   # +-1 voxel jitter margin stored around it
BIG = (CROP[0] + 2 * MARGIN, CROP[1] + 2 * MARGIN, CROP[2] + 2 * MARGIN)
OOF_COLUMNS = ["stem", "t", "z", "y", "x", "label", "fold", "p_cont", "p_par", "p_dau"]


def log(*args) -> None:
    print(time.strftime("%H:%M:%S"), *args, flush=True)


# ----------------------------------------------------------------------------------------------------------------------
# configuration
# ----------------------------------------------------------------------------------------------------------------------
def _env_flag(name: str, default: str = "0") -> bool:
    return os.environ.get(name, default).strip() not in ("", "0", "false", "False", "no")


def config_from_env(overrides: dict | None = None) -> dict:
    env = os.environ.get
    smoke = _env_flag("KDN_SMOKE")
    on_kaggle = Path("/kaggle/working").exists()
    cfg = {
        "smoke": smoke,
        "data_dir": env("KDN_DATA_DIR", ""),
        "work_dir": env("KDN_WORK_DIR", "/kaggle/working" if on_kaggle else str(Path.cwd() / "kdn_out")),
        "cache_dir": env("KDN_CACHE_DIR", ""),
        "splits_path": env("KDN_SPLITS", ""),
        "stems": [s.strip() for s in env("KDN_STEMS", "").split(",") if s.strip()],
        "max_videos": int(env("KDN_MAX_VIDEOS", "0") or 0),
        "smoke_videos": int(env("KDN_SMOKE_VIDEOS", "5")),
        "n_folds": int(env("KDN_FOLDS", "2" if smoke else "5")),
        "epochs": int(env("KDN_EPOCHS", "1" if smoke else "30")),
        "patience": int(env("KDN_PATIENCE", "6")),
        "batch": int(env("KDN_BATCH", "64")),
        "lr": float(env("KDN_LR", "1e-3")),
        "weight_decay": float(env("KDN_WD", "0.01")),
        "warmup_frac": float(env("KDN_WARMUP_FRAC", "0.03")),
        "epoch_samples": int(env("KDN_EPOCH_SAMPLES", "0")),   # 0 = number of training samples
        "eval_batch": int(env("KDN_EVAL_BATCH", "256")),
        "cont_per_video": int(env("KDN_CONT_PER_VIDEO", "60")),
        "hardneg_per_video": int(env("KDN_HARDNEG_PER_VIDEO", "40")),
        "cont_min_dist": int(env("KDN_CONT_MIN_DIST", "2")),
        "seed": int(env("KDN_SEED", "0")),
        "build_workers": int(env("KDN_BUILD_WORKERS", "0")),  # 0 = min(4, cpu count)
        "contract_checks": int(env("KDN_CONTRACT_CHECKS", "2")),  # exact contract checks per video
        "cache_dtype": env("KDN_CACHE_DTYPE", "float32"),
        "jitter": int(env("KDN_JITTER", "1")),
        "gain": [float(env("KDN_GAIN_LO", "0.8")), float(env("KDN_GAIN_HI", "1.2"))],
        "amp": _env_flag("KDN_AMP", "1"),
        "gpus": int(env("KDN_GPUS", "-1")),                   # -1 = all visible, 0 = CPU
        "inner_val": _env_flag("KDN_INNER_VAL"),              # 1 = early-stop on fold k+1, train on the other K-2
        "divnet_compare": _env_flag("KDN_DIVNET", "1"),
        "divnet_checkpoint": env("KDN_DIVNET_CKPT", ""),
        "keep_cache": _env_flag("KDN_KEEP_CACHE", "1"),
        "gate": {"child_um": 10.0, "v_um": 15.0, "sister_min_um": 3.0, "sister_max_um": 20.0,
                 "mid_um": 6.5, "cos_max": 0.25},
    }
    if overrides:
        cfg.update(overrides)
    return cfg


def resolve_data_dir(cfg: dict) -> Path:
    if cfg.get("data_dir"):
        return Path(cfg["data_dir"])
    candidates = [Path(f"/kaggle/input/competitions/{COMPETITION}"), Path(f"/kaggle/input/{COMPETITION}")]
    comp_dir = next((p for p in candidates if p.exists()), candidates[0])
    return comp_dir / "train"


def resolve_cache_dir(cfg: dict) -> Path:
    if cfg.get("cache_dir"):
        return Path(cfg["cache_dir"])
    for base in (Path("/kaggle/temp"), Path("/tmp")):
        if base.exists() and os.access(base, os.W_OK):
            return base / "kdn_cache"
    return Path(cfg["work_dir"]) / "_kdn_cache"


# ----------------------------------------------------------------------------------------------------------------------
# codecs: zstd (GT .geff arrays) and blosc (image chunks); several decoders so the kernel needs no extra wheels
# ----------------------------------------------------------------------------------------------------------------------
_ZSTD_FN = None


def _zstd_candidates():
    def via_zstandard(buf, size):
        import zstandard
        return zstandard.ZstdDecompressor().decompress(buf, max_output_size=max(size, 1))

    def via_numcodecs(buf, size):
        from numcodecs import Zstd
        return bytes(Zstd().decode(buf))

    def via_pyarrow(buf, size):
        import pyarrow as pa
        return pa.Codec("zstd").decompress(buf, decompressed_size=size, asbytes=True)

    def via_imagecodecs(buf, size):
        import imagecodecs
        return bytes(imagecodecs.zstd_decode(buf))

    return [via_zstandard, via_numcodecs, via_pyarrow, via_imagecodecs]


def zstd_decompress(buf: bytes, size: int) -> bytes:
    global _ZSTD_FN
    if _ZSTD_FN is not None:
        return _ZSTD_FN(buf, size)
    errors = []
    for fn in _zstd_candidates():
        try:
            out = fn(buf, size)
            if len(out) != size:
                raise ValueError(f"size {len(out)} != {size}")
            _ZSTD_FN = fn
            return out
        except Exception as exc:  # try the next decoder
            errors.append(f"{fn.__name__}: {type(exc).__name__}: {exc}")
    raise RuntimeError("no zstd decoder works: " + " | ".join(errors))


def _bitunshuffle(block: bytes, typesize: int) -> bytes:
    n = len(block) // typesize
    n8 = n - (n % 8)
    if n8 == 0:
        return block
    head = np.frombuffer(block, dtype=np.uint8, count=n8 * typesize).reshape(typesize * 8, n8 // 8)
    bits = np.unpackbits(head, axis=1, bitorder="little").reshape(typesize, 8, n8)
    elems = np.packbits(bits, axis=1, bitorder="little").reshape(typesize, n8)
    return elems.T.tobytes() + block[n8 * typesize:]


def blosc1_decompress_pure(raw: bytes) -> bytes:
    """Blosc-1 frame decoder (numpy + a zstd decoder); last-resort fallback, tested against blosc2 locally."""
    flags, typesize = raw[2], raw[3]
    nbytes, blocksize, _cbytes = struct.unpack("<iii", raw[4:16])
    if flags & 0x2:  # memcpyed
        return bytes(raw[16:16 + nbytes])
    compcode = flags >> 5
    if compcode != 4:
        raise RuntimeError(f"pure blosc decoder supports zstd only (compcode {compcode})")
    nblocks = (nbytes + blocksize - 1) // blocksize
    bstarts = struct.unpack(f"<{nblocks}i", raw[16:16 + 4 * nblocks])
    out = bytearray(nbytes)
    dont_split = bool(flags & 0x10)
    for i, start in enumerate(bstarts):
        bsize = min(blocksize, nbytes - i * blocksize)
        nstreams = 1 if (dont_split or typesize <= 1 or bsize < typesize) else typesize
        neb = bsize // nstreams
        pos, parts = start, []
        for _ in range(nstreams):
            csize = struct.unpack("<i", raw[pos:pos + 4])[0]
            pos += 4
            chunk = raw[pos:pos + csize]
            pos += csize
            parts.append(bytes(chunk) if csize == neb else zstd_decompress(bytes(chunk), neb))
        block = b"".join(parts)
        if flags & 0x4:
            block = _bitunshuffle(block, typesize)
        elif (flags & 0x1) and typesize > 1:
            n = bsize // typesize
            block = np.frombuffer(block, np.uint8, count=n * typesize).reshape(typesize, n).T.tobytes() + block[n * typesize:]
        out[i * blocksize:i * blocksize + bsize] = block
    return bytes(out)


_BLOSC_FN = None


def _blosc_candidates():
    def via_blosc2(raw):
        import blosc2
        return blosc2.decompress(raw)

    def via_numcodecs(raw):
        from numcodecs import Blosc
        return bytes(Blosc().decode(raw))

    def via_blosc(raw):
        import blosc
        return blosc.decompress(raw)

    return [via_blosc2, via_numcodecs, via_blosc, blosc1_decompress_pure]


def blosc_decompress(raw: bytes) -> bytes:
    global _BLOSC_FN
    nbytes = struct.unpack("<i", raw[4:8])[0]
    if _BLOSC_FN is not None:
        return _BLOSC_FN(raw)
    errors = []
    for fn in _blosc_candidates():
        try:
            out = fn(raw)
            if len(out) != nbytes:
                raise ValueError(f"size {len(out)} != {nbytes}")
            _BLOSC_FN = fn
            return out
        except Exception as exc:
            errors.append(f"{fn.__name__}: {type(exc).__name__}: {exc}")
    raise RuntimeError("no blosc decoder works: " + " | ".join(errors))


def decoder_names() -> dict:
    return {"zstd": getattr(_ZSTD_FN, "__name__", None), "blosc": getattr(_BLOSC_FN, "__name__", None)}


def _zarr_dtype(meta: dict) -> np.dtype:
    dtype = np.dtype(meta["data_type"])
    for codec in meta.get("codecs", []):
        if codec.get("name") == "bytes":
            endian = (codec.get("configuration") or {}).get("endian", "little")
            dtype = dtype.newbyteorder("<" if endian == "little" else ">")
    return dtype


def _decode_chunk(raw: bytes, meta: dict, chunk_shape: tuple) -> np.ndarray:
    dtype = _zarr_dtype(meta)
    size = int(np.prod(chunk_shape)) * dtype.itemsize
    data = raw
    codecs = meta.get("codecs", [])
    for codec in reversed(codecs):
        name = codec.get("name")
        if name == "bytes":
            continue
        if name == "zstd":
            data = zstd_decompress(data, size)
        elif name == "blosc":
            data = blosc_decompress(data)
        elif name == "gzip":
            data = zlib.decompress(data, 16 + zlib.MAX_WBITS)
        else:
            raise RuntimeError(f"unsupported zarr codec {name}")
    return np.frombuffer(data, dtype=dtype, count=int(np.prod(chunk_shape))).reshape(chunk_shape)


def read_zarr_array(path: Path) -> np.ndarray:
    """Whole zarr v3 array with a regular chunk grid, decoded with numpy (+ zstd/blosc); zarr is the fallback."""
    path = Path(path)
    try:
        meta = json.loads((path / "zarr.json").read_text())
        shape = tuple(int(v) for v in meta["shape"])
        chunks = tuple(int(v) for v in meta["chunk_grid"]["configuration"]["chunk_shape"])
        sep = ((meta.get("chunk_key_encoding") or {}).get("configuration") or {}).get("separator", "/")
        dtype = _zarr_dtype(meta)
        out = np.full(shape, meta.get("fill_value", 0) or 0, dtype=dtype.newbyteorder("="))
        grid = [range(math.ceil(s / c)) if s else range(0) for s, c in zip(shape, chunks)]
        for idx in np.ndindex(*[len(g) for g in grid]):
            key = sep.join(str(i) for i in idx)
            chunk_path = path / "c" / key if sep == "/" else path / f"c{sep}{key}"
            if not chunk_path.exists():
                continue
            chunk = _decode_chunk(chunk_path.read_bytes(), meta, chunks)
            sl = tuple(slice(i * c, min((i + 1) * c, s)) for i, c, s in zip(idx, chunks, shape))
            out[sl] = chunk[tuple(slice(0, x.stop - x.start) for x in sl)]
        return out
    except Exception as first:
        try:
            import zarr
            return np.asarray(zarr.open(str(path), mode="r")[...])
        except Exception:
            raise first


def read_gt_geff(path: Path) -> dict:
    """GT graph arrays: ids, t, z, y, x (voxel units of the full-resolution frame) and edges (E, 2) source->target."""
    path = Path(path)
    ids = read_zarr_array(path / "nodes" / "ids").astype(np.int64)
    props = path / "nodes" / "props"
    cols = {k: read_zarr_array(props / k / "values") for k in ("t", "z", "y", "x")}
    edges_path = path / "edges" / "ids"
    edges = read_zarr_array(edges_path).astype(np.int64).reshape(-1, 2) if edges_path.exists() else np.zeros((0, 2), np.int64)
    return {"ids": ids, "t": cols["t"].astype(np.int64), "z": cols["z"].astype(np.float64),
            "y": cols["y"].astype(np.float64), "x": cols["x"].astype(np.float64), "edges": edges}


class FrameReader:
    """read_test_frame of the exp39 notebook, parameterised by directory: one blosc chunk per frame, zarr fallback.

    Callable as read_frame(dataset, t, frame_cache); t is clamped to [0, T-1]; the frame is cached under the requested
    t so prepare_divnet_input's frame_cache.pop(frame_t) releases it."""

    def __init__(self, data_dir: Path) -> None:
        self.data_dir = Path(data_dir)
        self.meta: dict[str, tuple] = {}
        self.frames_read = 0

    def _meta(self, dataset: str) -> tuple:
        if dataset not in self.meta:
            meta = json.loads((self.data_dir / f"{dataset}.zarr" / "0" / "zarr.json").read_text())
            shape = tuple(int(v) for v in meta["shape"])
            chunks = tuple(int(v) for v in meta["chunk_grid"]["configuration"]["chunk_shape"])
            self.meta[dataset] = (meta, shape, chunks)
        return self.meta[dataset]

    def __call__(self, dataset: str, t: int, frame_cache: dict) -> np.ndarray:
        if t in frame_cache:
            return frame_cache[t]
        meta, shape, chunks = self._meta(dataset)
        index = max(0, min(int(t), shape[0] - 1))
        zarr_path = self.data_dir / f"{dataset}.zarr"
        frame = None
        if chunks[0] == 1 and tuple(chunks[1:]) == tuple(shape[1:]):
            try:
                raw = (zarr_path / "0" / "c" / str(index) / "0" / "0" / "0").read_bytes()
                frame = _decode_chunk(raw, meta, (1, *shape[1:]))[0].copy()
            except Exception:
                frame = None
        if frame is None:
            import zarr
            frame = np.asarray(zarr.open(str(zarr_path / "0"), mode="r")[index])
        self.frames_read += 1
        frame_cache[t] = frame
        return frame


# ----------------------------------------------------------------------------------------------------------------------
# labels
# ----------------------------------------------------------------------------------------------------------------------
def _stem_seed(stem: str, seed: int) -> list[int]:
    return [int(seed), zlib.crc32(stem.encode("utf-8"))]


def extract_labels(stem: str, geff_path: Path, cfg: dict) -> tuple[list[dict], dict]:
    g = read_gt_geff(geff_path)
    ids = g["ids"]
    n = len(ids)
    pos_of = {int(v): i for i, v in enumerate(ids)}
    t = g["t"]
    pts_um = np.stack([g["z"], g["y"], g["x"]], axis=1) * VOXEL_SCALE_UM
    children: list[list[int]] = [[] for _ in range(n)]
    parents: list[list[int]] = [[] for _ in range(n)]
    dropped_edges = 0
    for s, d in g["edges"]:
        a, b = pos_of.get(int(s)), pos_of.get(int(d))
        if a is None or b is None:
            dropped_edges += 1
            continue
        children[a].append(b)
        parents[b].append(a)
    outdeg = np.array([len(c) for c in children])
    indeg = np.array([len(p) for p in parents])
    is_parent = outdeg >= 2
    is_daughter = np.zeros(n, bool)
    for i in np.flatnonzero(is_parent):
        is_daughter[children[i]] = True
    ambiguous = is_parent & is_daughter
    division_node = is_parent | is_daughter

    # lineage distance (frames) to the nearest parent or daughter node among ancestors and descendants
    order = np.argsort(t, kind="stable")
    up = np.full(n, -np.inf)
    for i in order:
        if division_node[i]:
            up[i] = t[i]
        elif parents[i]:
            up[i] = max(up[p] for p in parents[i])
    down = np.full(n, np.inf)
    for i in order[::-1]:
        if division_node[i]:
            down[i] = t[i]
        elif children[i]:
            down[i] = min(down[c] for c in children[i])
    dist = np.minimum(t - up, down - t)
    cont_ok = (indeg == 1) & (outdeg == 1) & ~division_node & (dist >= int(cfg["cont_min_dist"]))

    # hard negatives: cont nodes in the insertion gate of an annotated one-child node (as u or as v)
    gate = cfg["gate"]
    by_t: dict[int, np.ndarray] = {}
    for tt in np.unique(t):
        by_t[int(tt)] = np.flatnonzero(t == tt)
    hard_role: dict[int, set] = {}
    gate_triples = 0
    for u in np.flatnonzero(outdeg == 1):
        c1 = children[u][0]
        if t[c1] != t[u] + 1:
            continue
        vec_c = pts_um[c1] - pts_um[u]
        n_c = float(np.linalg.norm(vec_c))
        if n_c > gate["child_um"]:
            continue
        cand = by_t.get(int(t[u]) + 1)
        if cand is None:
            continue
        cand = cand[cand != c1]
        vec_v = pts_um[cand] - pts_um[u]
        n_v = np.linalg.norm(vec_v, axis=1)
        sister = np.linalg.norm(pts_um[cand] - pts_um[c1], axis=1)
        mid = np.linalg.norm(0.5 * (pts_um[cand] + pts_um[c1]) - pts_um[u], axis=1)
        cos = (vec_v @ vec_c) / np.maximum(n_c * n_v, 1e-9)  # zero-length vector -> cos 0 (passes)
        ok = ((n_v <= gate["v_um"]) & (sister >= gate["sister_min_um"]) & (sister <= gate["sister_max_um"])
              & (mid <= gate["mid_um"]) & (cos <= gate["cos_max"]))
        for v in cand[ok]:
            gate_triples += 1
            if cont_ok[u]:
                hard_role.setdefault(int(u), set()).add("u")
            if cont_ok[v]:
                hard_role.setdefault(int(v), set()).add("v")

    rng = np.random.default_rng(_stem_seed(stem, cfg["seed"]))
    hard_nodes = sorted(hard_role)
    if len(hard_nodes) > cfg["hardneg_per_video"]:
        hard_nodes = sorted(rng.choice(hard_nodes, size=cfg["hardneg_per_video"], replace=False).tolist())
    hard_set = set(hard_nodes)
    cont_pool = [i for i in np.flatnonzero(cont_ok) if int(i) not in hard_set]
    if len(cont_pool) > cfg["cont_per_video"]:
        cont_pool = sorted(rng.choice(cont_pool, size=cfg["cont_per_video"], replace=False).tolist())

    rows: list[dict] = []

    def add(i: int, label: str, kind: str) -> None:
        rows.append({"stem": stem, "node_id": int(ids[i]), "t": int(t[i]), "z": float(g["z"][i]),
                     "y": float(g["y"][i]), "x": float(g["x"][i]), "label": label, "kind": kind})

    for i in np.flatnonzero(is_parent & ~ambiguous):
        add(int(i), "parent", "parent")
    for i in np.flatnonzero(is_daughter & ~ambiguous):
        add(int(i), "daughter", "daughter")
    for i in cont_pool:
        add(int(i), "cont", "cont")
    for i in hard_nodes:
        add(int(i), "cont", "hardneg_" + "".join(sorted(hard_role[i])))
    stats = {
        "stem": stem, "embryo": stem[:4], "gt_nodes": int(n), "gt_edges": int(len(g["edges"])),
        "dropped_edges": int(dropped_edges), "divisions": int(is_parent.sum()),
        "parent": int((is_parent & ~ambiguous).sum()), "daughter": int((is_daughter & ~ambiguous).sum()),
        "ambiguous_dropped": int(ambiguous.sum()), "cont_eligible": int(cont_ok.sum()),
        "cont": len(cont_pool), "gate_triples": int(gate_triples), "hardneg_candidates": len(hard_role),
        "hardneg": len(hard_nodes),
        "hardneg_u": sum(1 for i in hard_nodes if "u" in hard_role[i]),
        "hardneg_v": sum(1 for i in hard_nodes if "v" in hard_role[i]),
        "edge_dt_not_1": int(sum(1 for a in range(n) for b in children[a] if t[b] - t[a] != 1)),
    }
    return rows, stats


# ----------------------------------------------------------------------------------------------------------------------
# folds
# ----------------------------------------------------------------------------------------------------------------------
def make_splits_inline(n_div: dict[str, int], n_folds: int = 5, seed: int = 0) -> dict[str, int]:
    """Fallback only (used when no splits.json is available): 5 folds grouped by video, stratified by embryo prefix
    and division count, seed 0. Per embryo: videos in seeded random order, stable-sorted by division count (desc);
    division videos go greedily to the fold with the fewest divisions (then fewest videos), zero-division videos to
    the fold with the fewest videos of that embryo."""
    rng = np.random.default_rng(seed)
    folds: dict[str, int] = {}
    for emb in sorted({s[:4] for s in n_div}):
        vids = sorted(s for s in n_div if s[:4] == emb)
        vids = [vids[i] for i in rng.permutation(len(vids))]
        vids.sort(key=lambda s: -n_div[s])
        div_sum, cnt = [0] * n_folds, [0] * n_folds
        for s in vids:
            if n_div[s] > 0:
                f = min(range(n_folds), key=lambda k: (div_sum[k], cnt[k], k))
            else:
                f = min(range(n_folds), key=lambda k: (cnt[k], k))
            folds[s] = f
            div_sum[f] += n_div[s]
            cnt[f] += 1
    return folds


def load_splits(path: Path) -> dict[str, int] | None:
    if not path or not Path(path).is_file():
        return None
    d = json.loads(Path(path).read_text())
    if isinstance(d, dict) and isinstance(d.get("folds"), dict):
        return {str(k): int(v) for k, v in d["folds"].items()}
    if isinstance(d, dict) and isinstance(d.get("folds"), list):
        return {str(s): i for i, group in enumerate(d["folds"]) for s in group}
    if isinstance(d, dict) and all(isinstance(v, int) for v in d.values()):
        return {str(k): int(v) for k, v in d.items()}
    if isinstance(d, list):
        return {str(r["stem"]): int(r["fold"]) for r in d}
    raise ValueError(f"unrecognised splits format in {path}")


def _default_splits_path() -> Path | None:
    here = Path(__file__).resolve().parent
    for p in (here / "splits.json", here.parent / "splits.json"):
        if p.is_file():
            return p
    return None


# ----------------------------------------------------------------------------------------------------------------------
# step 1: prepare
# ----------------------------------------------------------------------------------------------------------------------
def step_prepare(cfg: dict) -> dict:
    import pandas as pd

    t0 = time.time()
    data_dir = resolve_data_dir(cfg)
    work = Path(cfg["work_dir"])
    work.mkdir(parents=True, exist_ok=True)
    cache = resolve_cache_dir(cfg)
    cache.mkdir(parents=True, exist_ok=True)
    all_stems = sorted(p.name[:-5] for p in data_dir.iterdir() if p.name.endswith(".geff")
                       and (data_dir / f"{p.name[:-5]}.zarr").exists())
    log(f"[prepare] data_dir={data_dir} videos with .zarr+.geff: {len(all_stems)}; cache={cache}; work={work}")
    if not all_stems:
        raise FileNotFoundError(f"no <stem>.zarr + <stem>.geff pairs under {data_dir}")

    # GT labels for every video (cheap: the .geff files are tiny); per-video try/except
    label_rows, stats_rows, n_div = [], [], {}
    for i, stem in enumerate(all_stems):
        ts = time.time()
        try:
            rows, st = extract_labels(stem, data_dir / f"{stem}.geff", cfg)
            st["label_secs"] = round(time.time() - ts, 3)
            label_rows.append(rows)
            stats_rows.append(st)
            n_div[stem] = st["divisions"]
        except Exception as exc:
            log(f"[prepare] {stem}: label extraction FAILED {type(exc).__name__}: {exc}")
            stats_rows.append({"stem": stem, "embryo": stem[:4], "error": f"{type(exc).__name__}: {exc}"})
            label_rows.append([])
            n_div[stem] = 0
    log(f"[prepare] GT parsed for {len(all_stems)} videos in {time.time() - t0:.1f} s; "
        f"divisions {sum(n_div.values())} in {sum(1 for v in n_div.values() if v)} videos; zstd decoder {decoder_names()['zstd']}")

    # folds
    splits_path = Path(cfg["splits_path"]) if cfg.get("splits_path") else _default_splits_path()
    splits = load_splits(splits_path) if splits_path else None
    splits_source = str(splits_path) if splits else "inline make_splits_inline(seed=0)"
    if splits is None:
        splits = make_splits_inline(n_div, 5, 0)
    missing = [s for s in all_stems if s not in splits]
    if missing:
        log(f"[prepare] WARNING {len(missing)} stems missing from splits -> inline fold rule for them")
        extra = make_splits_inline({s: n_div[s] for s in missing}, 5, 0)
        splits.update(extra)

    # video selection
    if cfg["stems"]:
        stems = [s for s in cfg["stems"] if s in set(all_stems)]
    else:
        limit = cfg["max_videos"] or 0
        if cfg["smoke"]:
            limit = min(limit, cfg["smoke_videos"]) if limit else cfg["smoke_videos"]
        if limit and limit < len(all_stems):
            by_emb = {e: sorted((s for s in all_stems if s[:4] == e), key=lambda s: (-n_div[s], s))
                      for e in sorted({s[:4] for s in all_stems})}
            stems, k = [], 0
            while len(stems) < limit:  # alternate embryos, division-rich videos first
                emb = sorted(by_emb, reverse=True)[k % len(by_emb)]
                if by_emb[emb]:
                    stems.append(by_emb[emb].pop(0))
                k += 1
                if not any(by_emb.values()):
                    break
            stems = sorted(stems)
        else:
            stems = list(all_stems)
    n_folds = int(cfg["n_folds"])
    orig_folds = sorted({splits[s] for s in splits})
    if n_folds == len(orig_folds) and len(stems) == len(all_stems):
        folds = {s: int(splits[s]) for s in stems}
    else:
        folds = {s: int(splits[s]) % n_folds for s in stems}
        if len({folds[s] for s in stems}) < min(n_folds, len(stems)):  # small subsets: deal round-robin
            ordered = sorted(stems, key=lambda s: (splits[s], s))
            folds = {s: i % n_folds for i, s in enumerate(ordered)}

    chosen = set(stems)
    rows = [r for stem, rs in zip(all_stems, label_rows) if stem in chosen for r in rs]
    meta = pd.DataFrame(rows, columns=["stem", "node_id", "t", "z", "y", "x", "label", "kind"])
    meta["label_idx"] = meta["label"].map(CLASS_INDEX).astype(int)
    meta["fold"] = meta["stem"].map(folds).astype(int)
    meta["embryo"] = meta["stem"].str[:4]
    meta = meta.sort_values(["stem", "t", "node_id", "kind"]).reset_index(drop=True)
    stats = pd.DataFrame(stats_rows)
    stats["selected"] = stats["stem"].isin(stems)
    stats["fold"] = stats["stem"].map(folds)
    (work / "splits_used.json").write_text(json.dumps(
        {"source": splits_source, "n_folds": n_folds, "folds": {s: folds[s] for s in stems}}, indent=1))
    log(f"[prepare] folds from {splits_source}; selected {len(stems)} videos, {len(meta)} samples; "
        f"per fold videos {dict(sorted(pd.Series(folds).value_counts().items()))}")
    _print_label_counts(meta)
    return {"data_dir": str(data_dir), "cache_dir": str(cache), "work_dir": str(work), "stems": stems,
            "folds": folds, "meta": meta, "stats": stats, "splits_source": splits_source,
            "timing": {"prepare_s": round(time.time() - t0, 1)}}


def _print_label_counts(meta) -> None:
    if meta.empty:
        log("[prepare] no labelled samples")
        return
    tab = meta.groupby(["embryo", "kind"]).size().unstack(fill_value=0)
    log("[prepare] samples by embryo x kind:\n" + tab.to_string())
    tab = meta.groupby(["fold", "label"]).size().unstack(fill_value=0)
    log("[prepare] samples by fold x label:\n" + tab.to_string())


# ----------------------------------------------------------------------------------------------------------------------
# step 2: crops
# ----------------------------------------------------------------------------------------------------------------------
def _big_cfg() -> dict:
    cfg = copy.deepcopy(DEFAULT_CFG)
    cfg["input"]["crop_z"] = BIG[0]
    cfg["input"]["crop_yx"] = BIG[1]
    return cfg


def _build_worker_init(n_threads: int) -> None:
    try:
        import blosc2
        blosc2.set_nthreads(max(1, n_threads))
    except Exception:
        pass
    try:
        import torch
        torch.set_num_threads(1)
    except Exception:
        pass


def crop_video(job: dict) -> dict:
    """All crops of one video: frames are read once (sorted by t), prepared frames kept for lags -1..+2 only."""
    t0 = time.time()
    stem = job["stem"]
    try:
        reader = FrameReader(Path(job["data_dir"]))
        prepared: dict = {}
        bundle = {"config": _big_cfg(), "prepared_frames": prepared}
        ref_bundle = {"config": copy.deepcopy(DEFAULT_CFG), "prepared_frames": prepared}
        frame_cache: dict = {}
        nodes = job["nodes"]
        out = np.empty((len(nodes), 4, *BIG), dtype=np.dtype(job["dtype"]))
        checked = 0
        inner = (slice(None), slice(MARGIN, MARGIN + CROP[0]), slice(MARGIN, MARGIN + CROP[1]),
                 slice(MARGIN, MARGIN + CROP[2]))
        for i in sorted(range(len(nodes)), key=lambda k: nodes[k][0]):
            tt, z, y, x = nodes[i]
            node = {"t": int(tt), "z": float(z), "y": float(y), "x": float(x)}
            sample = prepare_sample(stem, int(tt), node, bundle, frame_cache, reader)
            if sample.shape != (5, *BIG):
                raise RuntimeError(f"unexpected enlarged sample shape {sample.shape}")
            out[i] = sample[:4]
            if checked < job["checks"]:
                ref = prepare_sample(stem, int(tt), node, ref_bundle, frame_cache, reader)
                if ref.shape != (5, *CROP) or not np.array_equal(sample[inner], ref):
                    raise RuntimeError("inner crop != DivNet-contract sample (contract drift)")
                checked += 1
            for key in [k for k in prepared if k[1] < int(tt) - 1]:
                prepared.pop(key)
        prepared.clear()
        frame_cache.clear()
        return {"stem": stem, "crops": out, "secs": time.time() - t0, "error": None, "checked": checked,
                "frames_read": reader.frames_read, "decoders": decoder_names()}
    except Exception as exc:
        return {"stem": stem, "crops": None, "secs": time.time() - t0, "error": f"{type(exc).__name__}: {exc}",
                "checked": 0, "frames_read": 0, "decoders": decoder_names()}


def step_build_cache(cfg: dict, state: dict) -> dict:
    import multiprocessing as mp

    t0 = time.time()
    meta = state["meta"]
    cache = Path(state["cache_dir"])
    crops_path = cache / "crops.npy"
    dtype = np.dtype(cfg["cache_dtype"])
    n = len(meta)
    need = n * 4 * int(np.prod(BIG)) * dtype.itemsize
    try:
        import shutil
        free = shutil.disk_usage(cache).free
        log(f"[build] cache disk free {free / 2**30:.1f} GiB, needed {need / 2**30:.2f} GiB")
        if free < 1.2 * need and dtype == np.float32:
            dtype = np.dtype(np.float16)
            log("[build] WARNING not enough disk for float32 -> caching float16 (OOF inputs rounded to fp16)")
    except Exception:
        pass
    state["cache_dtype"] = dtype.name
    log(f"[build] {n} samples -> {crops_path} ({n * 4 * int(np.prod(BIG)) * dtype.itemsize / 2**30:.2f} GiB {dtype})")
    crops = np.lib.format.open_memmap(crops_path, mode="w+", dtype=dtype, shape=(max(n, 1), 4, *BIG))
    jobs = []
    for stem, grp in meta.groupby("stem", sort=True):
        jobs.append({"stem": stem, "data_dir": state["data_dir"], "dtype": dtype.name, "checks": cfg["contract_checks"],
                     "index": grp.index.to_numpy(),
                     "nodes": list(zip(grp["t"].tolist(), grp["z"].tolist(), grp["y"].tolist(), grp["x"].tolist()))})
    n_workers = cfg["build_workers"] or min(4, os.cpu_count() or 1)
    n_workers = max(1, min(n_workers, len(jobs) or 1))
    per_worker_threads = max(1, (os.cpu_count() or 1) // n_workers)
    valid = np.zeros(n, bool)
    video_rows = []
    lite = [{k: v for k, v in j.items() if k != "index"} for j in jobs]
    index_of = {j["stem"]: j["index"] for j in jobs}
    done = 0
    ctx = mp.get_context("spawn")
    with ctx.Pool(n_workers, initializer=_build_worker_init, initargs=(per_worker_threads,)) as pool:
        for res in pool.imap_unordered(crop_video, lite):
            done += 1
            stem = res["stem"]
            idx = index_of[stem]
            grp = meta.loc[idx]
            counts = grp["kind"].value_counts().to_dict()
            if res["error"] is None:
                crops[idx] = res["crops"]
                valid[idx] = True
                status = "ok"
            else:
                status = "FAILED " + res["error"]
            video_rows.append({"stem": stem, "crop_secs": round(res["secs"], 2), "frames_read": res["frames_read"],
                               "contract_checks": res["checked"], "crop_error": res["error"], "n_samples": len(idx)})
            log(f"[build] {done}/{len(jobs)} {stem} n={len(idx)} "
                + " ".join(f"{k}={v}" for k, v in sorted(counts.items()))
                + f" frames={res['frames_read']} {res['secs']:.1f}s decoders={res['decoders']} {status}")
    crops.flush()
    del crops
    meta = meta.copy()
    meta["valid"] = valid
    meta.to_csv(cache / "meta.csv", index_label="row")
    import pandas as pd
    vid = pd.DataFrame(video_rows)
    stats = state["stats"].merge(vid, on="stem", how="left") if not vid.empty else state["stats"]
    stats.to_csv(Path(state["work_dir"]) / "labels_summary.csv", index=False)
    state["meta"] = meta
    state["stats"] = stats
    secs = time.time() - t0
    state["timing"]["build_s"] = round(secs, 1)
    ok = vid[vid["crop_error"].isna()] if not vid.empty else vid
    per_video = float(ok["crop_secs"].mean()) if len(ok) else float("nan")
    state["timing"]["crop_s_per_video_mean"] = round(per_video, 2)
    log(f"[build] done in {secs:.1f} s with {n_workers} workers; valid samples {int(valid.sum())}/{n}; "
        f"videos failed {int(vid['crop_error'].notna().sum()) if not vid.empty else 0}; mean crop time per video "
        f"{per_video:.2f} s (single worker)")
    return state


# ----------------------------------------------------------------------------------------------------------------------
# step 3: training (subprocess per fold)
# ----------------------------------------------------------------------------------------------------------------------
def _src_paths() -> list[str]:
    import tools.daughternet as dn  # noqa: F401  (the tools package the kernel materialised or the repo copy)
    paths = [str(Path(__file__).resolve().parent), str(Path(dn.__file__).resolve().parent.parent)]
    return list(dict.fromkeys(paths))


def _fold_dir(work: Path, fold: int) -> Path:
    return work / "folds" / f"fold_{fold}"


def _pump(stream, prefix: str, sink) -> None:
    for line in iter(stream.readline, ""):
        print(prefix + line.rstrip(), flush=True)
        sink.write(line)
        sink.flush()
    stream.close()


def step_train(cfg: dict, state: dict) -> dict:
    import torch

    t0 = time.time()
    work = Path(state["work_dir"])
    cache = Path(state["cache_dir"])
    meta = state["meta"]
    folds = sorted(int(f) for f in meta.loc[meta["valid"], "fold"].unique())
    (work / "folds").mkdir(parents=True, exist_ok=True)
    cfg_path = cache / "kdn_cfg.json"
    cfg_path.write_text(json.dumps({**cfg, "_cache_dir": str(cache), "_work_dir": str(work)}, indent=1))
    n_gpu = torch.cuda.device_count() if cfg["gpus"] != 0 else 0
    if cfg["gpus"] > 0:
        n_gpu = min(n_gpu, cfg["gpus"])
    slots = [str(i) for i in range(n_gpu)] or ["cpu"]
    log(f"[train] folds {folds} on slots {slots} (subprocess per fold)")
    env_base = os.environ.copy()
    env_base["PYTHONPATH"] = os.pathsep.join(_src_paths() + [p for p in env_base.get("PYTHONPATH", "").split(os.pathsep) if p])
    env_base["PYTHONUNBUFFERED"] = "1"
    queue = list(folds)
    running: dict[str, tuple] = {}
    results: dict[int, int] = {}
    while queue or running:
        for slot in slots:
            if slot not in running and queue:
                fold = queue.pop(0)
                fdir = _fold_dir(work, fold)
                fdir.mkdir(parents=True, exist_ok=True)
                # a rerun into the same work dir must never pick up a previous run's fold model / OOF rows
                # (a fold that fails or never saves best.pt would otherwise load or merge stale files silently)
                for stale in ("best.pt", "last.pt", "oof.csv", "summary.json", "history.json", "train.log"):
                    (fdir / stale).unlink(missing_ok=True)
                env = dict(env_base)
                if slot != "cpu":
                    env["CUDA_VISIBLE_DEVICES"] = slot
                device = "cuda:0" if slot != "cpu" else "cpu"
                cmd = [sys.executable, str(Path(__file__).resolve()), "worker", "--cfg", str(cfg_path),
                       "--fold", str(fold), "--device", device, "--threads",
                       str(max(1, (os.cpu_count() or 1) // len(slots)))]
                sink = open(fdir / "train.log", "a", encoding="utf-8")
                proc = subprocess.Popen(cmd, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                                        encoding="utf-8", errors="replace")
                th = threading.Thread(target=_pump, args=(proc.stdout, f"[fold {fold}@{slot}] ", sink), daemon=True)
                th.start()
                running[slot] = (fold, proc, th, sink, time.time())
                log(f"[train] started fold {fold} on {slot} (pid {proc.pid})")
        time.sleep(1.0)
        for slot in list(running):
            fold, proc, th, sink, ts = running[slot]
            if proc.poll() is not None:
                th.join(timeout=10)
                sink.close()
                results[fold] = proc.returncode
                log(f"[train] fold {fold} finished rc={proc.returncode} in {time.time() - ts:.0f} s")
                del running[slot]
    state["train_rc"] = results
    state["timing"]["train_s"] = round(time.time() - t0, 1)
    failed = [f for f, rc in results.items() if rc != 0]
    if failed:
        log(f"[train] WARNING folds failed: {failed} (their videos get no OOF prediction)")
    return state


def _load_split(crops, rows: np.ndarray, inner_only: bool, dtype, device, torch, chunk: int = 1024):
    """Rows of the memmap -> one tensor on device; inner_only keeps the contract 16x32x32 crop."""
    if len(rows) == 0:
        shape = (0, 4, *CROP) if inner_only else (0, 4, *BIG)
        return torch.zeros(shape, dtype=dtype, device=device)
    parts = []
    for s in range(0, len(rows), chunk):
        block = np.asarray(crops[rows[s:s + chunk]], dtype=np.float32)
        if inner_only:
            block = block[:, :, MARGIN:MARGIN + CROP[0], MARGIN:MARGIN + CROP[1], MARGIN:MARGIN + CROP[2]]
        parts.append(torch.from_numpy(np.ascontiguousarray(block)).to(device=device, dtype=dtype))
    return torch.cat(parts)


def augment_batch(xb, rng: np.random.Generator, cfg: dict, marker, torch):
    """(B, 4, 18, 34, 34) image crops -> (B, 5, 16, 32, 32): +-jitter crop (image shifts, marker stays centred), then
    random flips on z/y/x and a random k*90 degree rotation in yx applied to all 5 channels, then a per-sample
    intensity gain on the 4 image channels (clipped back to the contract range [-0.5, 6])."""
    b = xb.shape[0]
    dev = xb.device
    j = int(cfg["jitter"])
    offs = torch.from_numpy(rng.integers(MARGIN - j, MARGIN + j + 1, size=(b, 3))).to(dev)
    zi = offs[:, 0, None] + torch.arange(CROP[0], device=dev)
    yi = offs[:, 1, None] + torch.arange(CROP[1], device=dev)
    xi = offs[:, 2, None] + torch.arange(CROP[2], device=dev)
    bi = torch.arange(b, device=dev)[:, None, None, None]
    img = xb[bi, :, zi[:, :, None, None], yi[:, None, :, None], xi[:, None, None, :]]  # (B, 16, 32, 32, 4)
    img = img.permute(0, 4, 1, 2, 3).float()
    x = torch.cat([img, marker.to(dev).expand(b, 1, *CROP)], dim=1)
    flips = torch.from_numpy(rng.random((b, 3)) < 0.5).to(dev)
    for axis, dim in enumerate((2, 3, 4)):
        m = flips[:, axis].view(b, 1, 1, 1, 1)
        x = torch.where(m, x.flip(dim), x)
    rot = rng.integers(0, 4, size=b)
    for k in (1, 2, 3):
        sel = np.flatnonzero(rot == k)
        if len(sel):
            idx = torch.from_numpy(sel).to(dev)
            x[idx] = torch.rot90(x[idx], k, dims=(3, 4))
    gain = torch.from_numpy(rng.uniform(cfg["gain"][0], cfg["gain"][1], size=(b, 1, 1, 1, 1)).astype(np.float32)).to(dev)
    x[:, :4] = (x[:, :4] * gain).clamp(-0.5, 6.0)
    return x


def contract_marker(torch):
    sigma = tuple(float(v) for v in DEFAULT_CFG["input"]["marker_sigma"])
    return torch.from_numpy(_marker(CROP, sigma)).view(1, 1, *CROP)


def predict_inner(model, x_inner, marker, batch: int, torch) -> np.ndarray:
    """Softmax (N, 3) for contract samples (image inner crops + centred marker), fp32 like score_nodes."""
    out = []
    model.eval()
    with torch.no_grad():
        for s in range(0, x_inner.shape[0], batch):
            xb = x_inner[s:s + batch].float()
            x = torch.cat([xb, marker.to(xb.device).expand(xb.shape[0], 1, *CROP)], dim=1)
            out.append(torch.softmax(model(x).float(), dim=1).cpu().numpy())
    return np.concatenate(out) if out else np.zeros((0, len(CLASSES)), np.float32)


def auc_score(scores, positive) -> float | None:
    import pandas as pd
    s = np.asarray(scores, dtype=np.float64)
    y = np.asarray(positive, dtype=bool)
    n1, n0 = int(y.sum()), int((~y).sum())
    if n1 == 0 or n0 == 0:
        return None
    ranks = pd.Series(s).rank(method="average").to_numpy()
    return float((ranks[y].sum() - n1 * (n1 + 1) / 2.0) / (n1 * n0))


def _val_metrics(probs: np.ndarray, y: np.ndarray) -> dict:
    if len(y) == 0:
        return {"n": 0}
    p_true = np.clip(probs[np.arange(len(y)), y], 1e-7, 1.0)
    ce = -np.log(p_true)
    present = [c for c in range(len(CLASSES)) if (y == c).any()]
    macro = float(np.mean([ce[y == c].mean() for c in present]))
    res = {"n": int(len(y)), "macro_ce": macro, "ce": float(ce.mean()),
           "acc": float((probs.argmax(1) == y).mean())}
    for c, name in enumerate(CLASSES):
        res[f"n_{name}"] = int((y == c).sum())
        res[f"ce_{name}"] = float(ce[y == c].mean()) if (y == c).any() else None
        res[f"auc_{name}"] = auc_score(probs[:, c], y == c)
    return res


def _atomic_save(obj, path: Path, torch) -> None:
    tmp = path.with_suffix(path.suffix + ".tmp")
    torch.save(obj, tmp)
    os.replace(tmp, path)


def train_fold(cfg: dict, fold: int, device_str: str, threads: int = 0) -> dict:
    import pandas as pd
    import torch

    t_start = time.time()
    if threads:
        torch.set_num_threads(int(threads))
    cache = Path(cfg["_cache_dir"])
    work = Path(cfg["_work_dir"])
    fdir = _fold_dir(work, fold)
    fdir.mkdir(parents=True, exist_ok=True)
    device = torch.device(device_str)
    use_cuda = device.type == "cuda"
    if use_cuda:
        torch.backends.cudnn.benchmark = True
    seed = int(cfg["seed"]) * 1000 + int(fold)
    torch.manual_seed(seed)
    rng = np.random.default_rng(seed)
    meta = pd.read_csv(cache / "meta.csv", index_col="row")
    crops = np.load(cache / "crops.npy", mmap_mode="r")
    meta = meta[meta["valid"].astype(bool)]
    n_folds = int(cfg["n_folds"])
    if cfg.get("inner_val") and n_folds < 3:
        log(f"fold {fold}: KDN_INNER_VAL needs >= 3 folds (have {n_folds}) -> early stopping on the OOF fold")
        cfg = {**cfg, "inner_val": False}
    if cfg.get("inner_val"):
        es_fold = (fold + 1) % n_folds
        train_mask = ~meta["fold"].isin([fold, es_fold])
        es_mask = meta["fold"] == es_fold
    else:
        es_fold = fold
        train_mask = meta["fold"] != fold
        es_mask = meta["fold"] == fold
    oof_mask = meta["fold"] == fold
    tr_rows = meta.index[train_mask].to_numpy()
    es_rows = meta.index[es_mask].to_numpy()
    oof_rows = meta.index[oof_mask].to_numpy()
    y_tr = meta.loc[tr_rows, "label_idx"].to_numpy().astype(np.int64)
    y_es = meta.loc[es_rows, "label_idx"].to_numpy().astype(np.int64)
    log(f"fold {fold}: device {device} train {len(tr_rows)} {np.bincount(y_tr, minlength=3).tolist()} | "
        f"early-stop fold {es_fold}: {len(es_rows)} {np.bincount(y_es, minlength=3).tolist()} | oof {len(oof_rows)}")
    if len(tr_rows) == 0:
        raise RuntimeError("no training samples")
    store_dtype = torch.float16 if use_cuda else torch.float32
    t_load = time.time()
    x_tr = _load_split(crops, tr_rows, False, store_dtype, device, torch)
    x_es = _load_split(crops, es_rows, True, torch.float32, device, torch)
    y_tr_t = torch.from_numpy(y_tr).to(device)
    log(f"fold {fold}: loaded crops in {time.time() - t_load:.1f} s (train {tuple(x_tr.shape)} {x_tr.dtype})")
    marker = contract_marker(torch).to(device)

    ckpt_cfg = copy.deepcopy(DEFAULT_CFG)
    ckpt_cfg["classes"] = list(CLASSES)
    ckpt_cfg["train"] = {k: cfg[k] for k in ("epochs", "patience", "batch", "lr", "weight_decay", "warmup_frac",
                                               "epoch_samples", "jitter", "gain", "seed", "cont_per_video",
                                               "hardneg_per_video", "cont_min_dist", "inner_val", "n_folds")}
    ckpt_cfg["train"].update({"fold": fold, "early_stop_fold": es_fold, "kernel": "biohub-l2-kdn-train"})
    model = build_model(ckpt_cfg).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=cfg["lr"], weight_decay=cfg["weight_decay"])
    classes_present = [c for c in range(len(CLASSES)) if (y_tr == c).any()]
    pools = {c: np.flatnonzero(y_tr == c) for c in classes_present}
    epoch_samples = int(cfg["epoch_samples"]) or len(tr_rows)
    steps_per_epoch = max(1, math.ceil(epoch_samples / cfg["batch"]))
    total_steps = steps_per_epoch * int(cfg["epochs"])
    warmup = max(1, int(cfg["warmup_frac"] * total_steps))

    def lr_at(step: int) -> float:
        if step < warmup:
            return cfg["lr"] * (step + 1) / warmup
        prog = (step - warmup) / max(1, total_steps - warmup)
        return cfg["lr"] * 0.5 * (1.0 + math.cos(math.pi * min(1.0, prog)))

    use_amp = bool(cfg["amp"]) and use_cuda
    try:
        scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
    except Exception:  # older torch
        scaler = torch.cuda.amp.GradScaler(enabled=use_amp)
    best, best_epoch, bad, step = float("inf"), -1, 0, 0
    history = []
    for epoch in range(int(cfg["epochs"])):
        te = time.time()
        model.train()
        cls = rng.choice(classes_present, size=epoch_samples)
        order = np.empty(epoch_samples, dtype=np.int64)
        for c in classes_present:
            sel = np.flatnonzero(cls == c)
            order[sel] = rng.choice(pools[c], size=len(sel), replace=True)
        rng.shuffle(order)
        run_loss, seen = torch.zeros((), device=device), 0
        for s in range(0, epoch_samples, cfg["batch"]):
            idx = order[s:s + cfg["batch"]]
            idx_t = torch.from_numpy(idx).to(device)
            xb = augment_batch(x_tr[idx_t], rng, cfg, marker, torch)
            yb = y_tr_t[idx_t]
            for group in opt.param_groups:
                group["lr"] = lr_at(step)
            opt.zero_grad(set_to_none=True)
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
                logits = model(xb)
            loss = torch.nn.functional.cross_entropy(logits.float(), yb)
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
            run_loss += loss.detach() * len(idx)
            seen += len(idx)
            step += 1
        train_loss = float(run_loss) / max(1, seen)
        if use_cuda:
            torch.cuda.synchronize()
        train_secs = time.time() - te
        probs = predict_inner(model, x_es, marker, cfg["eval_batch"], torch)
        vm = _val_metrics(probs, y_es)
        val_loss = vm.get("macro_ce", float("nan")) if vm["n"] else float("nan")
        improved = vm["n"] > 0 and val_loss < best
        state_cpu = {k: v.detach().cpu() for k, v in model.state_dict().items()}
        payload = {"model_state": state_cpu, "config": ckpt_cfg, "fold": int(fold), "epoch": int(epoch),
                   "val_loss": float(val_loss)}
        _atomic_save(payload, fdir / "last.pt", torch)
        if improved or (vm["n"] == 0):
            best, best_epoch, bad = (val_loss if vm["n"] else best), epoch, 0
            _atomic_save(payload, fdir / "best.pt", torch)
        else:
            bad += 1
        rec = {"epoch": epoch, "train_loss": train_loss, "val_loss": val_loss, "lr": lr_at(step - 1),
               "secs": round(time.time() - te, 1), "train_secs": round(train_secs, 1),
               "samples_per_s": round(seen / max(1e-9, train_secs), 1),
               **{f"val_{k}": v for k, v in vm.items()}}
        history.append(rec)
        (fdir / "history.json").write_text(json.dumps(history, indent=1))
        auc_txt = " ".join(f"auc_{n}={vm.get('auc_' + n) if vm.get('auc_' + n) is None else round(vm['auc_' + n], 3)}"
                           for n in CLASSES)
        log(f"fold {fold} epoch {epoch}: train {train_loss:.4f} val(macro CE) {val_loss:.4f} "
            f"{'*' if improved else ' '} {auc_txt} {rec['secs']}s (train {rec['samples_per_s']} samples/s)")
        if vm["n"] and bad >= int(cfg["patience"]):
            log(f"fold {fold}: early stop at epoch {epoch} (best {best_epoch}, val {best:.4f})")
            break
    del x_tr
    if not (fdir / "best.pt").is_file():
        raise RuntimeError(f"fold {fold}: no best.pt written in this run (epochs={cfg['epochs']}, history "
                           f"{[h['val_loss'] for h in history]}) -> no OOF for this fold")
    # OOF prediction with best.pt through the shared loader (same path the submission patch uses)
    bundle = load_daughternet([fdir / "best.pt"], device=str(device))
    x_oof = _load_split(crops, oof_rows, True, torch.float32, device, torch)
    p_oof = predict_inner(bundle["models"][0], x_oof, marker, cfg["eval_batch"], torch)
    oof = pd.DataFrame({"row": oof_rows, "p_cont": p_oof[:, 0], "p_par": p_oof[:, 1], "p_dau": p_oof[:, 2]})
    oof.to_csv(fdir / "oof.csv", index=False)
    y_oof = meta.loc[oof_rows, "label_idx"].to_numpy().astype(np.int64)
    om = _val_metrics(p_oof, y_oof)
    summary = {"fold": fold, "best_epoch": best_epoch, "best_val_loss": best, "epochs_run": len(history),
               "early_stop_fold": es_fold, "n_train": int(len(tr_rows)), "n_oof": int(len(oof_rows)),
               "train_counts": np.bincount(y_tr, minlength=3).tolist(), "oof_metrics": om,
               "secs": round(time.time() - t_start, 1), "device": str(device)}
    (fdir / "summary.json").write_text(json.dumps(summary, indent=1))
    log(f"fold {fold}: OOF {len(oof_rows)} samples macro CE {om.get('macro_ce')} "
        f"auc_parent {om.get('auc_parent')} auc_daughter {om.get('auc_daughter')}; total {summary['secs']} s")
    return summary


def worker_main(argv: list[str]) -> int:
    import argparse

    ap = argparse.ArgumentParser()
    ap.add_argument("--cfg", required=True)
    ap.add_argument("--fold", type=int, required=True)
    ap.add_argument("--device", default="cpu")
    ap.add_argument("--threads", type=int, default=0)
    a = ap.parse_args(argv)
    cfg = json.loads(Path(a.cfg).read_text())
    train_fold(cfg, a.fold, a.device, a.threads)
    return 0


# ----------------------------------------------------------------------------------------------------------------------
# step 4: OOF merge, readout, DivNet comparison
# ----------------------------------------------------------------------------------------------------------------------
TOP_FRACS = (("top0.1%", 0.001), ("top1%", 0.01), ("top5%", 0.05))


def _rank_block(scores: np.ndarray, positive: np.ndarray) -> dict:
    n = len(scores)
    n_pos = int(positive.sum())
    base = n_pos / n if n else None
    out = {"n": int(n), "n_pos": n_pos, "base_rate": base, "auc": auc_score(scores, positive)}
    order = np.argsort(-scores, kind="stable")
    for name, frac in TOP_FRACS:
        if n == 0:
            out[name] = None
            continue
        k = max(1, int(math.ceil(frac * n)))
        tp = int(positive[order[:k]].sum())
        fp = k - tp
        prec = tp / k
        tpr = tp / n_pos if n_pos else None
        fpr = fp / (n - n_pos) if n - n_pos else None
        lr_plus = (tpr / fpr) if (tpr is not None and fpr) else None
        out[name] = {"k": k, "tp": tp, "precision": prec, "lift": (prec / base) if base else None,
                     "lr_plus": lr_plus, "lr_plus_infinite": bool(fpr == 0 and tp > 0)}
    return out


def _find_divnet_checkpoint(cfg: dict) -> Path | None:
    if cfg.get("divnet_checkpoint"):
        p = Path(cfg["divnet_checkpoint"])
        return p if p.is_file() else None
    for p in _checkpoint_candidates():
        if Path(p).is_file():
            return Path(p)
    root = Path("/kaggle/input")
    if root.exists():  # pruned walk: a plain rglob would descend into every competition .zarr chunk tree
        found = []
        for dirpath, dirnames, files in os.walk(root, followlinks=True):
            depth = Path(dirpath).relative_to(root).parts
            dirnames[:] = [] if len(depth) >= 6 else sorted(
                d for d in dirnames if not d.endswith((".zarr", ".geff")) and not d.startswith("."))
            if "best_overall.pt" in files and "divnet" in dirpath.lower():
                found.append(Path(dirpath) / "best_overall.pt")
        if found:
            return sorted(found)[0]
    return None


def step_oof(cfg: dict, state: dict) -> dict:
    import pandas as pd
    import torch

    t0 = time.time()
    work = Path(state["work_dir"])
    cache = Path(state["cache_dir"])
    meta = pd.read_csv(cache / "meta.csv", index_col="row")
    meta = meta[meta["valid"].astype(bool)].copy()
    for col in ("p_cont", "p_par", "p_dau"):
        meta[col] = np.nan
    fold_summaries = {}
    for fold in sorted(meta["fold"].unique()):
        fdir = _fold_dir(work, int(fold))
        f_oof = fdir / "oof.csv"
        if f_oof.is_file():
            o = pd.read_csv(f_oof).set_index("row")
            meta.loc[o.index, ["p_cont", "p_par", "p_dau"]] = o[["p_cont", "p_par", "p_dau"]].to_numpy()
        if (fdir / "summary.json").is_file():
            fold_summaries[int(fold)] = json.loads((fdir / "summary.json").read_text())
    has_p = meta["p_par"].notna()
    log(f"[oof] predictions for {int(has_p.sum())}/{len(meta)} samples")

    divnet_info = {"enabled": bool(cfg["divnet_compare"])}
    if cfg["divnet_compare"]:
        try:
            ckpt = _find_divnet_checkpoint(cfg)
            if ckpt is None:
                raise FileNotFoundError("best_overall.pt not found")
            bundle = load_divnet_mitosis_model(ckpt)
            dev = bundle["device"]
            crops = np.load(cache / "crops.npy", mmap_mode="r")
            marker = contract_marker(torch).to(dev)
            rows = meta.index.to_numpy()
            ps = []
            for s in range(0, len(rows), 1024):
                x_in = _load_split(crops, rows[s:s + 1024], True, torch.float32, dev, torch)
                with torch.no_grad():
                    for b in range(0, x_in.shape[0], cfg["eval_batch"]):
                        xb = x_in[b:b + cfg["eval_batch"]]
                        x = torch.cat([xb, marker.expand(xb.shape[0], 1, *CROP)], dim=1)
                        ps.append(torch.sigmoid(bundle["model"](x)).flatten().float().cpu().numpy())
            meta["p_divnet"] = np.concatenate(ps) if ps else np.zeros(0)
            del crops, bundle
            divnet_info.update({"checkpoint": str(ckpt), "in_sample": True,
                                "note": "public giorgosi DivNet best_overall.pt was trained on these 199 train videos "
                                        "(4 folds, best fold model): its numbers are IN-SAMPLE, DaughterNet's are OOF"})
            log(f"[oof] DivNet scored {len(meta)} samples with {ckpt}")
        except Exception as exc:
            divnet_info["error"] = f"{type(exc).__name__}: {exc}"
            log(f"[oof] DivNet comparison skipped: {divnet_info['error']}")

    out = meta.reset_index(drop=True)
    extra = [c for c in ("node_id", "kind", "embryo", "label_idx", "p_divnet") if c in out.columns]
    out[OOF_COLUMNS + extra].to_csv(work / "oof.csv", index=False)

    readout = {
        "kernel": "biohub-l2-kdn-train", "classes": list(CLASSES), "n_samples": int(len(out)),
        "n_with_oof": int(out["p_par"].notna().sum()), "splits_source": state.get("splits_source"),
        "definitions": {
            "base_rate": "fraction of the embryo's labelled samples that carry the class (sampling-dependent: cont is "
                         "subsampled to <= cont_per_video + hardneg_per_video per video)",
            "precision": "fraction of the class among the top-k samples ranked by that class probability",
            "lift": "precision / base_rate",
            "lr_plus": "likelihood ratio at the cutoff = TPR / FPR = (tp/P) / (fp/N); null when fp = 0",
            "auc": "one-vs-rest ROC AUC of the class probability",
        },
        "counts": {e: g.groupby("kind").size().to_dict() for e, g in out.groupby("embryo")},
        "per_embryo": {}, "hardneg": {}, "folds": fold_summaries, "divnet_comparison": divnet_info,
        "timing": state.get("timing", {}), "train_rc": state.get("train_rc"),
        "config": {k: v for k, v in cfg.items() if not k.startswith("_")},
    }
    scored = out[out["p_par"].notna()]
    groups = [(e, g) for e, g in scored.groupby("embryo")] + [("all", scored)]
    prob_col = {"cont": "p_cont", "parent": "p_par", "daughter": "p_dau"}
    for emb, g in groups:
        readout["per_embryo"][emb] = {
            name: _rank_block(g[prob_col[name]].to_numpy(), (g["label"] == name).to_numpy()) for name in CLASSES}
        hu = g[g["kind"].str.startswith("hardneg") & g["kind"].str.contains("u")]
        hv = g[g["kind"].str.startswith("hardneg") & g["kind"].str.contains("v")]
        par, dau = g[g["label"] == "parent"], g[g["label"] == "daughter"]
        readout["hardneg"][emb] = {
            "n_hardneg_u": int(len(hu)), "n_hardneg_v": int(len(hv)),
            "auc_p_par_parent_vs_hardneg_u": auc_score(np.r_[par["p_par"], hu["p_par"]],
                                                       np.r_[np.ones(len(par)), np.zeros(len(hu))]),
            "auc_p_dau_daughter_vs_hardneg_v": auc_score(np.r_[dau["p_dau"], hv["p_dau"]],
                                                         np.r_[np.ones(len(dau)), np.zeros(len(hv))]),
        }
        if "p_divnet" in g.columns and divnet_info.get("in_sample"):
            sub = g[g["label"].isin(["parent", "cont"])]
            sub_rand = g[g["kind"].isin(["parent", "cont"])]
            y = (sub["label"] == "parent").to_numpy()
            y_r = (sub_rand["kind"] == "parent").to_numpy()
            divnet_info.setdefault("per_embryo", {})[emb] = {
                "subset": "parent vs cont (random cont + hard negatives)",
                "divnet_in_sample": _rank_block(sub["p_divnet"].to_numpy(), y),
                "daughternet_oof_p_par": _rank_block(sub["p_par"].to_numpy(), y),
                "random_cont_only": {"divnet_in_sample_auc": auc_score(sub_rand["p_divnet"].to_numpy(), y_r),
                                     "daughternet_oof_auc": auc_score(sub_rand["p_par"].to_numpy(), y_r)},
            }
    state["timing"]["oof_s"] = round(time.time() - t0, 1)
    readout["timing"] = state["timing"]
    (work / "oof_readout.json").write_text(json.dumps(readout, indent=1, default=_json_default))
    log(f"[oof] wrote {work / 'oof.csv'} ({len(out)} rows) and {work / 'oof_readout.json'}")
    _print_readout(readout)
    if not cfg.get("keep_cache"):
        import shutil
        shutil.rmtree(cache, ignore_errors=True)
    state["readout"] = readout
    return state


def _json_default(value):
    if isinstance(value, np.integer):
        return int(value)
    if isinstance(value, np.floating):
        return float(value)
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, Path):
        return str(value)
    return str(value)


def _print_readout(readout: dict) -> None:
    for emb, block in readout["per_embryo"].items():
        for name, r in block.items():
            tops = " ".join(f"{k}:{r[k]['tp']}/{r[k]['k']}" for k, _ in TOP_FRACS if r.get(k))
            auc = None if r["auc"] is None else round(r["auc"], 4)
            base = "n/a" if r["base_rate"] is None else f"{r['base_rate']:.4f}"
            log(f"[readout] {emb:5s} {name:8s} n_pos={r['n_pos']:5d} base={base} AUC={auc} {tops}")
    for emb, r in readout.get("divnet_comparison", {}).get("per_embryo", {}).items():
        a, b = r["divnet_in_sample"]["auc"], r["daughternet_oof_p_par"]["auc"]
        log(f"[readout] {emb:5s} parent-vs-cont AUC DivNet(in-sample)={a} DaughterNet(OOF)={b}")


def step_verify(cfg: dict, state: dict, per_fold: int = 2) -> dict:
    """End-to-end contract check on a few OOF samples per fold: raw frames -> tools.daughternet.score_nodes with the
    fold's best.pt (the submission-patch path) must reproduce oof.csv, and DivNet's divnet_score_division must
    reproduce p_divnet. Differences are only fp rounding (float32 cache) or fp16 rounding (float16 cache)."""
    import pandas as pd
    import torch
    from tools.divnet_infer import divnet_score_division

    work = Path(state["work_dir"])
    oof = pd.read_csv(work / "oof.csv")
    oof = oof[oof["p_par"].notna()]
    reader = FrameReader(Path(state["data_dir"]))
    report = {"checked": 0, "max_abs_diff_daughternet": 0.0, "max_abs_diff_divnet": None, "rows": []}
    divnet_bundle = None
    if "p_divnet" in oof.columns and oof["p_divnet"].notna().any():
        ckpt = _find_divnet_checkpoint(cfg)
        divnet_bundle = load_divnet_mitosis_model(ckpt) if ckpt else None
        report["max_abs_diff_divnet"] = 0.0
    for fold, g in oof.groupby("fold"):
        best = _fold_dir(work, int(fold)) / "best.pt"
        if not best.is_file():
            continue
        bundle = load_daughternet([best], device="cpu")
        # prefer parents/daughters so the check covers the classes that matter
        pick = pd.concat([g[g["label"] != "cont"].head(per_fold), g[g["label"] == "cont"].head(1)])
        for _, r in pick.iterrows():
            node = {"t": int(r["t"]), "z": float(r["z"]), "y": float(r["y"]), "x": float(r["x"])}
            bundle["prepared_frames"] = {}
            p = score_nodes(r["stem"], [node], bundle, {}, reader, batch=1)[0]
            diff = float(np.abs(p - r[["p_cont", "p_par", "p_dau"]].to_numpy(dtype=np.float64)).max())
            report["max_abs_diff_daughternet"] = max(report["max_abs_diff_daughternet"], diff)
            row = {"fold": int(fold), "stem": r["stem"], "t": int(r["t"]), "label": r["label"], "diff": diff}
            if divnet_bundle is not None:
                divnet_bundle["prepared_frames"] = {}
                q = divnet_score_division(r["stem"], int(r["t"]), node, None, None, divnet_bundle, {}, reader)
                d2 = abs(q - float(r["p_divnet"]))
                report["max_abs_diff_divnet"] = max(report["max_abs_diff_divnet"], d2)
                row["diff_divnet"] = d2
            report["rows"].append(row)
            report["checked"] += 1
    # OOF p come from the fold worker (cuDNN on GPU, batch 256); the re-score runs on CPU one node at a time, so
    # only fp rounding separates them; a contract drift (crop shift, normalisation) moves p far more than this.
    tol = 1e-3 if state.get("cache_dtype", cfg["cache_dtype"]) == "float32" else 2e-2
    report["tolerance"] = tol
    report["pass"] = bool(report["checked"] > 0 and report["max_abs_diff_daughternet"] <= tol
                          and (report["max_abs_diff_divnet"] is None or report["max_abs_diff_divnet"] <= tol))
    (work / "contract_check.json").write_text(json.dumps(report, indent=1, default=_json_default))
    log(f"[verify] {report['checked']} samples re-scored from raw frames via score_nodes: max |dp| DaughterNet "
        f"{report['max_abs_diff_daughternet']:.2e}, DivNet {report['max_abs_diff_divnet']} -> "
        f"{'PASS' if report['pass'] else 'FAIL'} (tol {tol})")
    state["verify"] = report
    return state


def run_all(cfg: dict) -> dict:
    state = step_prepare(cfg)
    state = step_build_cache(cfg, state)
    state = step_train(cfg, state)
    state = step_oof(cfg, state)
    state = step_verify(cfg, state)
    return state


if __name__ == "__main__":
    if len(sys.argv) > 1 and sys.argv[1] == "worker":
        sys.exit(worker_main(sys.argv[2:]))
    run_all(config_from_env())


In [ ]:
%%writefile /kaggle/working/kdn_src/splits.json
{
 "folds": {
  "44b6_0113de3b": 4,
  "44b6_0b24845f": 1,
  "44b6_0c582fdc": 2,
  "44b6_0db75fae": 2,
  "44b6_12dfb391": 4,
  "44b6_144b256d": 3,
  "44b6_1574802b": 0,
  "44b6_18ced818": 1,
  "44b6_1d530831": 4,
  "44b6_24264f12": 4,
  "44b6_267148e4": 2,
  "44b6_2a2eff9f": 3,
  "44b6_2f31fc2f": 1,
  "44b6_33b596bf": 2,
  "44b6_341df25f": 1,
  "44b6_3a861e03": 4,
  "44b6_3bb3690f": 2,
  "44b6_40c45f5a": 4,
  "44b6_415c0a3a": 3,
  "44b6_53f95252": 0,
  "44b6_551a5dba": 4,
  "44b6_5740d24b": 0,
  "44b6_587a1e22": 2,
  "44b6_5f15d135": 0,
  "44b6_668e0cc7": 3,
  "44b6_66f9292d": 3,
  "44b6_706092f0": 0,
  "44b6_71a4179f": 4,
  "44b6_74d0c52e": 0,
  "44b6_7a302da0": 4,
  "44b6_7e557709": 4,
  "44b6_808952d6": 1,
  "44b6_81c256f0": 1,
  "44b6_87bba6c4": 1,
  "44b6_8cc6506c": 2,
  "44b6_8f5ab931": 4,
  "44b6_8f9ecab4": 3,
  "44b6_90724892": 4,
  "44b6_949adeb1": 4,
  "44b6_95029e92": 1,
  "44b6_97725144": 0,
  "44b6_996155de": 1,
  "44b6_9be80b04": 2,
  "44b6_9bfa6a0a": 4,
  "44b6_a21120c2": 4,
  "44b6_a2bb48bb": 4,
  "44b6_aaf8b0ea": 1,
  "44b6_abf82518": 0,
  "44b6_b2c44266": 4,
  "44b6_c15fded2": 4,
  "44b6_c50204e0": 4,
  "44b6_c771cb04": 0,
  "44b6_c8e2a523": 4,
  "44b6_c96cfa10": 4,
  "44b6_cf2536e8": 3,
  "44b6_cf8fed6b": 3,
  "44b6_d29c9ab2": 4,
  "44b6_d2f34f90": 2,
  "44b6_d5e7d891": 4,
  "44b6_d754aa59": 1,
  "44b6_d78e09d9": 4,
  "44b6_db3c847b": 1,
  "44b6_ddf577ad": 3,
  "44b6_deabac95": 0,
  "44b6_e28840c6": 3,
  "44b6_e29f0176": 2,
  "44b6_e31261b4": 3,
  "44b6_e35b117d": 0,
  "44b6_e57ff5c6": 0,
  "44b6_eb2880fc": 3,
  "44b6_f28707c6": 2,
  "6bba_05b6850b": 2,
  "6bba_05db0fb1": 3,
  "6bba_062c8d37": 0,
  "6bba_07477033": 3,
  "6bba_07e24132": 0,
  "6bba_085bf656": 0,
  "6bba_09961292": 2,
  "6bba_0c7fa718": 2,
  "6bba_0e7c0d07": 1,
  "6bba_12665c0e": 1,
  "6bba_13f19531": 1,
  "6bba_15403b9a": 0,
  "6bba_1d0d8384": 2,
  "6bba_1ebfb80d": 2,
  "6bba_1f58c2f6": 2,
  "6bba_207c6aaf": 1,
  "6bba_20852818": 1,
  "6bba_2312ac41": 4,
  "6bba_23af9eeb": 1,
  "6bba_2540cd90": 2,
  "6bba_2646afc7": 3,
  "6bba_268e1230": 3,
  "6bba_2819ca14": 3,
  "6bba_283bf9f1": 4,
  "6bba_312f0dc3": 4,
  "6bba_32db13fc": 3,
  "6bba_337b1b3a": 1,
  "6bba_372c8cb8": 0,
  "6bba_3a1849c2": 1,
  "6bba_3abfe10a": 4,
  "6bba_3c5691b6": 0,
  "6bba_3db54e20": 0,
  "6bba_3fda6b25": 1,
  "6bba_43fea39d": 2,
  "6bba_474be664": 0,
  "6bba_48816121": 3,
  "6bba_4aa14c89": 2,
  "6bba_4f99ce20": 4,
  "6bba_4ffd3da3": 2,
  "6bba_55b7eebe": 0,
  "6bba_55c70843": 0,
  "6bba_57b7cc1e": 3,
  "6bba_5a4d9360": 4,
  "6bba_5b28472a": 1,
  "6bba_5c039895": 4,
  "6bba_5c824876": 2,
  "6bba_5dfe9ad1": 4,
  "6bba_5f89039d": 3,
  "6bba_61dd1e0d": 0,
  "6bba_61ecbe65": 2,
  "6bba_6321a359": 4,
  "6bba_6479435d": 3,
  "6bba_67ebd073": 2,
  "6bba_6ca87370": 3,
  "6bba_6db0e9b4": 3,
  "6bba_6feb10f0": 1,
  "6bba_6feeb0b1": 0,
  "6bba_705ec2c9": 1,
  "6bba_718b21f9": 2,
  "6bba_74686d6a": 3,
  "6bba_767a1e17": 0,
  "6bba_76db78c1": 3,
  "6bba_784a78c9": 1,
  "6bba_786893ac": 3,
  "6bba_789f8168": 2,
  "6bba_78a7bd97": 0,
  "6bba_7af54fde": 3,
  "6bba_7b5d3b2c": 3,
  "6bba_7d3058ae": 1,
  "6bba_7f87b3d8": 2,
  "6bba_80d12824": 2,
  "6bba_825bd1c6": 3,
  "6bba_87289e13": 3,
  "6bba_8b7818bf": 1,
  "6bba_907271db": 0,
  "6bba_91951b3a": 4,
  "6bba_96833384": 0,
  "6bba_969618f6": 0,
  "6bba_971fa5e0": 4,
  "6bba_9a41d029": 4,
  "6bba_9e23430b": 4,
  "6bba_a5e926bb": 2,
  "6bba_a90a0b9c": 1,
  "6bba_ab78413d": 0,
  "6bba_acd782a8": 2,
  "6bba_ae82a791": 3,
  "6bba_aeee7805": 4,
  "6bba_af149c94": 1,
  "6bba_afb141ff": 4,
  "6bba_b1ae37b9": 3,
  "6bba_b204cac7": 1,
  "6bba_b329af44": 1,
  "6bba_b693381b": 3,
  "6bba_bb9f20c3": 0,
  "6bba_bbb708ca": 0,
  "6bba_c27cba08": 0,
  "6bba_c328f2fd": 2,
  "6bba_c73a1d11": 0,
  "6bba_cdcfe533": 4,
  "6bba_cf35214c": 1,
  "6bba_cff5865f": 3,
  "6bba_d0fc38b5": 4,
  "6bba_d1acb6ff": 4,
  "6bba_d2b9fc0c": 3,
  "6bba_d3da753b": 2,
  "6bba_d5eae175": 1,
  "6bba_d6ecebbb": 1,
  "6bba_d82a4fc6": 1,
  "6bba_debd7bfa": 0,
  "6bba_df673a83": 1,
  "6bba_e16ffc58": 3,
  "6bba_e5e44988": 3,
  "6bba_ebdf3b34": 2,
  "6bba_ebff6e76": 4,
  "6bba_ed9377fd": 0,
  "6bba_edf14583": 4,
  "6bba_eebc57a5": 0,
  "6bba_ef7b4f7e": 2,
  "6bba_f17befbc": 1,
  "6bba_f1fde7e0": 3,
  "6bba_f20478e9": 2,
  "6bba_f4ae811c": 1,
  "6bba_f8ffd5e7": 0,
  "6bba_fbc898dc": 0,
  "6bba_fc516dc6": 2,
  "6bba_fc5f39dc": 1,
  "6bba_fc83837d": 2,
  "6bba_fe670320": 2
 },
 "selection": [
  "6bba_05db0fb1",
  "6bba_07477033",
  "6bba_085bf656",
  "6bba_0c7fa718",
  "6bba_12665c0e",
  "6bba_15403b9a",
  "6bba_1ebfb80d",
  "6bba_207c6aaf",
  "6bba_2312ac41",
  "6bba_2540cd90",
  "6bba_268e1230",
  "6bba_283bf9f1",
  "6bba_32db13fc",
  "6bba_372c8cb8",
  "6bba_3abfe10a",
  "6bba_3db54e20",
  "6bba_43fea39d",
  "6bba_48816121",
  "6bba_4f99ce20",
  "6bba_55b7eebe",
  "6bba_57b7cc1e",
  "6bba_5b28472a",
  "6bba_5c824876",
  "6bba_5f89039d",
  "6bba_61ecbe65",
  "6bba_6479435d",
  "6bba_6ca87370",
  "6bba_6feb10f0",
  "6bba_705ec2c9",
  "6bba_74686d6a",
  "6bba_76db78c1",
  "6bba_786893ac",
  "6bba_78a7bd97",
  "6bba_7b5d3b2c",
  "6bba_7f87b3d8",
  "6bba_825bd1c6",
  "6bba_8b7818bf",
  "6bba_91951b3a",
  "6bba_969618f6",
  "6bba_9a41d029",
  "6bba_a5e926bb",
  "6bba_ab78413d",
  "6bba_ae82a791",
  "6bba_af149c94",
  "6bba_b1ae37b9",
  "6bba_b329af44",
  "6bba_bb9f20c3",
  "6bba_c27cba08",
  "6bba_c73a1d11",
  "6bba_cf35214c",
  "6bba_d0fc38b5",
  "6bba_d2b9fc0c",
  "6bba_d5eae175",
  "6bba_d82a4fc6",
  "6bba_df673a83",
  "6bba_e5e44988",
  "6bba_ebff6e76",
  "6bba_edf14583",
  "6bba_ef7b4f7e",
  "6bba_f1fde7e0",
  "6bba_f4ae811c",
  "6bba_fbc898dc",
  "6bba_fc5f39dc",
  "6bba_fe670320"
 ],
 "confirmation": [
  "44b6_0113de3b",
  "44b6_0b24845f",
  "44b6_0c582fdc",
  "44b6_0db75fae",
  "44b6_12dfb391",
  "44b6_144b256d",
  "44b6_1574802b",
  "44b6_18ced818",
  "44b6_1d530831",
  "44b6_24264f12",
  "44b6_267148e4",
  "44b6_2a2eff9f",
  "44b6_2f31fc2f",
  "44b6_33b596bf",
  "44b6_341df25f",
  "44b6_3a861e03",
  "44b6_3bb3690f",
  "44b6_40c45f5a",
  "44b6_415c0a3a",
  "44b6_53f95252",
  "44b6_551a5dba",
  "44b6_5740d24b",
  "44b6_587a1e22",
  "44b6_5f15d135",
  "44b6_668e0cc7",
  "44b6_66f9292d",
  "44b6_706092f0",
  "44b6_71a4179f",
  "44b6_74d0c52e",
  "44b6_7a302da0",
  "44b6_7e557709",
  "44b6_808952d6",
  "44b6_81c256f0",
  "44b6_87bba6c4",
  "44b6_8cc6506c",
  "44b6_8f5ab931",
  "44b6_8f9ecab4",
  "44b6_90724892",
  "44b6_949adeb1",
  "44b6_95029e92",
  "44b6_97725144",
  "44b6_996155de",
  "44b6_9be80b04",
  "44b6_9bfa6a0a",
  "44b6_a21120c2",
  "44b6_a2bb48bb",
  "44b6_aaf8b0ea",
  "44b6_abf82518",
  "44b6_b2c44266",
  "44b6_c15fded2",
  "44b6_c50204e0",
  "44b6_c771cb04",
  "44b6_c8e2a523",
  "44b6_c96cfa10",
  "44b6_cf2536e8",
  "44b6_cf8fed6b",
  "44b6_d29c9ab2",
  "44b6_d2f34f90",
  "44b6_d5e7d891",
  "44b6_d754aa59",
  "44b6_d78e09d9",
  "44b6_db3c847b",
  "44b6_ddf577ad",
  "44b6_deabac95",
  "44b6_e28840c6",
  "44b6_e29f0176",
  "44b6_e31261b4",
  "44b6_e35b117d",
  "44b6_e57ff5c6",
  "44b6_eb2880fc",
  "44b6_f28707c6",
  "6bba_05b6850b",
  "6bba_062c8d37",
  "6bba_07e24132",
  "6bba_09961292",
  "6bba_0e7c0d07",
  "6bba_13f19531",
  "6bba_1d0d8384",
  "6bba_1f58c2f6",
  "6bba_20852818",
  "6bba_23af9eeb",
  "6bba_2646afc7",
  "6bba_2819ca14",
  "6bba_312f0dc3",
  "6bba_337b1b3a",
  "6bba_3a1849c2",
  "6bba_3c5691b6",
  "6bba_3fda6b25",
  "6bba_474be664",
  "6bba_4aa14c89",
  "6bba_4ffd3da3",
  "6bba_55c70843",
  "6bba_5a4d9360",
  "6bba_5c039895",
  "6bba_5dfe9ad1",
  "6bba_61dd1e0d",
  "6bba_6321a359",
  "6bba_67ebd073",
  "6bba_6db0e9b4",
  "6bba_6feeb0b1",
  "6bba_718b21f9",
  "6bba_767a1e17",
  "6bba_784a78c9",
  "6bba_789f8168",
  "6bba_7af54fde",
  "6bba_7d3058ae",
  "6bba_80d12824",
  "6bba_87289e13",
  "6bba_907271db",
  "6bba_96833384",
  "6bba_971fa5e0",
  "6bba_9e23430b",
  "6bba_a90a0b9c",
  "6bba_acd782a8",
  "6bba_aeee7805",
  "6bba_afb141ff",
  "6bba_b204cac7",
  "6bba_b693381b",
  "6bba_bbb708ca",
  "6bba_c328f2fd",
  "6bba_cdcfe533",
  "6bba_cff5865f",
  "6bba_d1acb6ff",
  "6bba_d3da753b",
  "6bba_d6ecebbb",
  "6bba_debd7bfa",
  "6bba_e16ffc58",
  "6bba_ebdf3b34",
  "6bba_ed9377fd",
  "6bba_eebc57a5",
  "6bba_f17befbc",
  "6bba_f20478e9",
  "6bba_f8ffd5e7",
  "6bba_fc516dc6",
  "6bba_fc83837d"
 ],
 "divisions_per_video": {
  "44b6_0113de3b": 0,
  "44b6_0b24845f": 0,
  "44b6_0c582fdc": 0,
  "44b6_0db75fae": 0,
  "44b6_12dfb391": 1,
  "44b6_144b256d": 0,
  "44b6_1574802b": 0,
  "44b6_18ced818": 0,
  "44b6_1d530831": 0,
  "44b6_24264f12": 0,
  "44b6_267148e4": 1,
  "44b6_2a2eff9f": 1,
  "44b6_2f31fc2f": 0,
  "44b6_33b596bf": 0,
  "44b6_341df25f": 1,
  "44b6_3a861e03": 0,
  "44b6_3bb3690f": 0,
  "44b6_40c45f5a": 0,
  "44b6_415c0a3a": 0,
  "44b6_53f95252": 0,
  "44b6_551a5dba": 0,
  "44b6_5740d24b": 0,
  "44b6_587a1e22": 1,
  "44b6_5f15d135": 1,
  "44b6_668e0cc7": 0,
  "44b6_66f9292d": 0,
  "44b6_706092f0": 1,
  "44b6_71a4179f": 0,
  "44b6_74d0c52e": 1,
  "44b6_7a302da0": 1,
  "44b6_7e557709": 0,
  "44b6_808952d6": 0,
  "44b6_81c256f0": 0,
  "44b6_87bba6c4": 0,
  "44b6_8cc6506c": 0,
  "44b6_8f5ab931": 0,
  "44b6_8f9ecab4": 0,
  "44b6_90724892": 0,
  "44b6_949adeb1": 0,
  "44b6_95029e92": 0,
  "44b6_97725144": 0,
  "44b6_996155de": 1,
  "44b6_9be80b04": 1,
  "44b6_9bfa6a0a": 0,
  "44b6_a21120c2": 2,
  "44b6_a2bb48bb": 0,
  "44b6_aaf8b0ea": 1,
  "44b6_abf82518": 0,
  "44b6_b2c44266": 0,
  "44b6_c15fded2": 0,
  "44b6_c50204e0": 2,
  "44b6_c771cb04": 0,
  "44b6_c8e2a523": 1,
  "44b6_c96cfa10": 0,
  "44b6_cf2536e8": 0,
  "44b6_cf8fed6b": 0,
  "44b6_d29c9ab2": 0,
  "44b6_d2f34f90": 2,
  "44b6_d5e7d891": 2,
  "44b6_d754aa59": 1,
  "44b6_d78e09d9": 0,
  "44b6_db3c847b": 0,
  "44b6_ddf577ad": 0,
  "44b6_deabac95": 1,
  "44b6_e28840c6": 2,
  "44b6_e29f0176": 0,
  "44b6_e31261b4": 0,
  "44b6_e35b117d": 0,
  "44b6_e57ff5c6": 0,
  "44b6_eb2880fc": 1,
  "44b6_f28707c6": 0,
  "6bba_05b6850b": 0,
  "6bba_05db0fb1": 3,
  "6bba_062c8d37": 1,
  "6bba_07477033": 0,
  "6bba_07e24132": 2,
  "6bba_085bf656": 1,
  "6bba_09961292": 4,
  "6bba_0c7fa718": 0,
  "6bba_0e7c0d07": 1,
  "6bba_12665c0e": 3,
  "6bba_13f19531": 0,
  "6bba_15403b9a": 0,
  "6bba_1d0d8384": 1,
  "6bba_1ebfb80d": 0,
  "6bba_1f58c2f6": 0,
  "6bba_207c6aaf": 1,
  "6bba_20852818": 3,
  "6bba_2312ac41": 2,
  "6bba_23af9eeb": 0,
  "6bba_2540cd90": 0,
  "6bba_2646afc7": 0,
  "6bba_268e1230": 1,
  "6bba_2819ca14": 1,
  "6bba_283bf9f1": 0,
  "6bba_312f0dc3": 0,
  "6bba_32db13fc": 1,
  "6bba_337b1b3a": 2,
  "6bba_372c8cb8": 0,
  "6bba_3a1849c2": 0,
  "6bba_3abfe10a": 1,
  "6bba_3c5691b6": 2,
  "6bba_3db54e20": 1,
  "6bba_3fda6b25": 1,
  "6bba_43fea39d": 0,
  "6bba_474be664": 1,
  "6bba_48816121": 5,
  "6bba_4aa14c89": 0,
  "6bba_4f99ce20": 2,
  "6bba_4ffd3da3": 3,
  "6bba_55b7eebe": 1,
  "6bba_55c70843": 0,
  "6bba_57b7cc1e": 3,
  "6bba_5a4d9360": 0,
  "6bba_5b28472a": 0,
  "6bba_5c039895": 3,
  "6bba_5c824876": 1,
  "6bba_5dfe9ad1": 0,
  "6bba_5f89039d": 0,
  "6bba_61dd1e0d": 0,
  "6bba_61ecbe65": 1,
  "6bba_6321a359": 1,
  "6bba_6479435d": 0,
  "6bba_67ebd073": 1,
  "6bba_6ca87370": 1,
  "6bba_6db0e9b4": 0,
  "6bba_6feb10f0": 0,
  "6bba_6feeb0b1": 0,
  "6bba_705ec2c9": 1,
  "6bba_718b21f9": 0,
  "6bba_74686d6a": 2,
  "6bba_767a1e17": 0,
  "6bba_76db78c1": 0,
  "6bba_784a78c9": 1,
  "6bba_786893ac": 3,
  "6bba_789f8168": 0,
  "6bba_78a7bd97": 2,
  "6bba_7af54fde": 2,
  "6bba_7b5d3b2c": 0,
  "6bba_7d3058ae": 2,
  "6bba_7f87b3d8": 3,
  "6bba_80d12824": 1,
  "6bba_825bd1c6": 0,
  "6bba_87289e13": 1,
  "6bba_8b7818bf": 2,
  "6bba_907271db": 1,
  "6bba_91951b3a": 0,
  "6bba_96833384": 0,
  "6bba_969618f6": 3,
  "6bba_971fa5e0": 0,
  "6bba_9a41d029": 2,
  "6bba_9e23430b": 1,
  "6bba_a5e926bb": 0,
  "6bba_a90a0b9c": 0,
  "6bba_ab78413d": 2,
  "6bba_acd782a8": 0,
  "6bba_ae82a791": 0,
  "6bba_aeee7805": 1,
  "6bba_af149c94": 0,
  "6bba_afb141ff": 4,
  "6bba_b1ae37b9": 0,
  "6bba_b204cac7": 2,
  "6bba_b329af44": 1,
  "6bba_b693381b": 0,
  "6bba_bb9f20c3": 3,
  "6bba_bbb708ca": 0,
  "6bba_c27cba08": 0,
  "6bba_c328f2fd": 2,
  "6bba_c73a1d11": 0,
  "6bba_cdcfe533": 4,
  "6bba_cf35214c": 0,
  "6bba_cff5865f": 0,
  "6bba_d0fc38b5": 0,
  "6bba_d1acb6ff": 1,
  "6bba_d2b9fc0c": 2,
  "6bba_d3da753b": 1,
  "6bba_d5eae175": 0,
  "6bba_d6ecebbb": 0,
  "6bba_d82a4fc6": 0,
  "6bba_debd7bfa": 4,
  "6bba_df673a83": 4,
  "6bba_e16ffc58": 1,
  "6bba_e5e44988": 0,
  "6bba_ebdf3b34": 2,
  "6bba_ebff6e76": 0,
  "6bba_ed9377fd": 0,
  "6bba_edf14583": 0,
  "6bba_eebc57a5": 0,
  "6bba_ef7b4f7e": 2,
  "6bba_f17befbc": 0,
  "6bba_f1fde7e0": 0,
  "6bba_f20478e9": 1,
  "6bba_f4ae811c": 0,
  "6bba_f8ffd5e7": 2,
  "6bba_fbc898dc": 0,
  "6bba_fc516dc6": 0,
  "6bba_fc5f39dc": 2,
  "6bba_fc83837d": 0,
  "6bba_fe670320": 2
 },
 "summary": {
  "total": {
   "videos": 199,
   "divisions": 151,
   "videos_with_divisions": 87,
   "44b6": {
    "videos": 71,
    "divisions": 26,
    "videos_with_divisions": 21
   },
   "6bba": {
    "videos": 128,
    "divisions": 125,
    "videos_with_divisions": 66
   }
  },
  "per_embryo": {
   "44b6": {
    "videos": 71,
    "divisions": 26,
    "videos_with_divisions": 21
   },
   "6bba": {
    "videos": 128,
    "divisions": 125,
    "videos_with_divisions": 66
   }
  },
  "per_fold": {
   "0": {
    "videos": 39,
    "divisions": 30,
    "videos_with_divisions": 18,
    "44b6": {
     "videos": 12,
     "divisions": 4,
     "videos_with_divisions": 4
    },
    "6bba": {
     "videos": 27,
     "divisions": 26,
     "videos_with_divisions": 14
    },
    "selection_videos": 12,
    "confirmation_videos": 27,
    "division_count_histogram": {
     "0": 21,
     "1": 10,
     "2": 5,
     "3": 2,
     "4": 1
    }
   },
   "1": {
    "videos": 39,
    "divisions": 30,
    "videos_with_divisions": 18,
    "44b6": {
     "videos": 12,
     "divisions": 4,
     "videos_with_divisions": 4
    },
    "6bba": {
     "videos": 27,
     "divisions": 26,
     "videos_with_divisions": 14
    },
    "selection_videos": 14,
    "confirmation_videos": 25,
    "division_count_histogram": {
     "0": 21,
     "1": 10,
     "2": 5,
     "3": 2,
     "4": 1
    }
   },
   "2": {
    "videos": 38,
    "divisions": 30,
    "videos_with_divisions": 18,
    "44b6": {
     "videos": 11,
     "divisions": 5,
     "videos_with_divisions": 4
    },
    "6bba": {
     "videos": 27,
     "divisions": 25,
     "videos_with_divisions": 14
    },
    "selection_videos": 10,
    "confirmation_videos": 28,
    "division_count_histogram": {
     "0": 20,
     "1": 10,
     "2": 5,
     "3": 2,
     "4": 1
    }
   },
   "3": {
    "videos": 39,
    "divisions": 30,
    "videos_with_divisions": 16,
    "44b6": {
     "videos": 12,
     "divisions": 4,
     "videos_with_divisions": 3
    },
    "6bba": {
     "videos": 27,
     "divisions": 26,
     "videos_with_divisions": 13
    },
    "selection_videos": 19,
    "confirmation_videos": 20,
    "division_count_histogram": {
     "0": 23,
     "1": 8,
     "2": 4,
     "3": 3,
     "5": 1
    }
   },
   "4": {
    "videos": 44,
    "divisions": 31,
    "videos_with_divisions": 17,
    "44b6": {
     "videos": 24,
     "divisions": 9,
     "videos_with_divisions": 6
    },
    "6bba": {
     "videos": 20,
     "divisions": 22,
     "videos_with_divisions": 11
    },
    "selection_videos": 9,
    "confirmation_videos": 35,
    "division_count_histogram": {
     "0": 27,
     "1": 8,
     "2": 6,
     "3": 1,
     "4": 2
    }
   }
  },
  "per_split": {
   "selection": {
    "videos": 64,
    "divisions": 64,
    "videos_with_divisions": 32,
    "44b6": {
     "videos": 0,
     "divisions": 0,
     "videos_with_divisions": 0
    },
    "6bba": {
     "videos": 64,
     "divisions": 64,
     "videos_with_divisions": 32
    }
   },
   "confirmation": {
    "videos": 135,
    "divisions": 87,
    "videos_with_divisions": 55,
    "44b6": {
     "videos": 71,
     "divisions": 26,
     "videos_with_divisions": 21
    },
    "6bba": {
     "videos": 64,
     "divisions": 61,
     "videos_with_divisions": 34
    }
   }
  },
  "division_count_histogram": {
   "44b6": {
    "0": 50,
    "1": 16,
    "2": 5
   },
   "6bba": {
    "0": 62,
    "1": 30,
    "2": 20,
    "3": 10,
    "4": 5,
    "5": 1
   }
  },
  "shared_annotation": {
   "min_shared_nodes": 8,
   "min_shared_ids": 4,
   "links": 62,
   "links_per_embryo": {
    "44b6": 62,
    "6bba": 0
   },
   "shared_gt_nodes": 2429,
   "shared_gt_divisions": 2,
   "groups": 11,
   "grouped_videos": 64,
   "group_sizes": [
    24,
    11,
    6,
    4,
    3,
    3,
    3,
    3,
    3,
    2,
    2
   ],
   "cross_fold_links": 0,
   "selection_confirmation_links": 0
  }
 },
 "meta": {
  "generator": "tools/make_splits.py",
  "seed": 20260923,
  "n_folds": 5,
  "division_definition": "GT node with >= 2 outgoing GT edges (== tracksdata dividing_nodes())",
  "reader": "tracksdata IndexedRXGraph.from_geff (same as tools/score_geffs.py)",
  "fold_method": "videos that share GT annotations (overlapping space-time crops: >= 8 identical GT nodes or >= 4 shared node ids at one (dt,dz,dy,dx) offset) are merged into one unit (leak_groups) that sits in one fold; units placed greedily then improved by best moves/swaps on a weighted squared deviation of videos + divisions per fold (w=2.0), per (embryo, fold) (w=1.0) and video counts per (embryo, min(n_div,3)) stratum (w=1.0); sha256(seed:...) ordering, no RNG",
  "selection_rule": "6bba stems with odd index (0-based) in sorted 6bba order",
  "confirmation_rule": "6bba even-index stems + all 44b6 stems",
  "leak_groups": [
   [
    "44b6_0113de3b",
    "44b6_12dfb391",
    "44b6_1d530831",
    "44b6_24264f12",
    "44b6_3a861e03",
    "44b6_40c45f5a",
    "44b6_551a5dba",
    "44b6_71a4179f",
    "44b6_7a302da0",
    "44b6_7e557709",
    "44b6_8f5ab931",
    "44b6_90724892",
    "44b6_949adeb1",
    "44b6_9bfa6a0a",
    "44b6_a21120c2",
    "44b6_a2bb48bb",
    "44b6_b2c44266",
    "44b6_c15fded2",
    "44b6_c50204e0",
    "44b6_c8e2a523",
    "44b6_c96cfa10",
    "44b6_d29c9ab2",
    "44b6_d5e7d891",
    "44b6_d78e09d9"
   ],
   [
    "44b6_144b256d",
    "44b6_2a2eff9f",
    "44b6_415c0a3a",
    "44b6_668e0cc7",
    "44b6_66f9292d",
    "44b6_8f9ecab4",
    "44b6_cf2536e8",
    "44b6_cf8fed6b",
    "44b6_ddf577ad",
    "44b6_e31261b4",
    "44b6_eb2880fc"
   ],
   [
    "44b6_341df25f",
    "44b6_808952d6",
    "44b6_81c256f0",
    "44b6_996155de",
    "44b6_aaf8b0ea",
    "44b6_db3c847b"
   ],
   [
    "44b6_53f95252",
    "44b6_5f15d135",
    "44b6_97725144",
    "44b6_e57ff5c6"
   ],
   [
    "44b6_0b24845f",
    "44b6_2f31fc2f",
    "44b6_87bba6c4"
   ],
   [
    "44b6_0c582fdc",
    "44b6_33b596bf",
    "44b6_f28707c6"
   ],
   [
    "44b6_0db75fae",
    "44b6_267148e4",
    "44b6_587a1e22"
   ],
   [
    "44b6_1574802b",
    "44b6_abf82518",
    "44b6_deabac95"
   ],
   [
    "44b6_3bb3690f",
    "44b6_8cc6506c",
    "44b6_d2f34f90"
   ],
   [
    "44b6_5740d24b",
    "44b6_74d0c52e"
   ],
   [
    "44b6_95029e92",
    "44b6_d754aa59"
   ]
  ],
  "shared_annotation_links": [
   {
    "a": "44b6_0113de3b",
    "b": "44b6_c8e2a523",
    "shared_nodes": 14,
    "shared_divisions": 0,
    "offset_tzyx_a_minus_b": [
     45,
     50,
     11,
     148
    ]
   },
   {
    "a": "44b6_0b24845f",
    "b": "44b6_2f31fc2f",
    "shared_nodes": 10,
    "shared_divisions": 0,
    "offset_tzyx_a_minus_b": [
     -49,
     12,
     88,
     209
    ]
   },
   {
    "a": "44b6_0c582fdc",
    "b": "44b6_33b596bf",
    "shared_nodes": 7,
    "shared_divisions": 0,
    "offset_tzyx_a_minus_b": [
     -44,
     20,
     221,
     88
    ]
   },
   {
    "a": "44b6_0db75fae",
    "b": "44b6_267148e4",
    "shared_nodes": 9,
    "shared_divisions": 0,
    "offset_tzyx_a_minus_b": [
     -28,
     -22,
     -51,
     240
    ]
   },
   {
    "a": "44b6_0db75fae",
    "b": "44b6_587a1e22",
    "shared_nodes": 25,
    "shared_divisions": 0,
    "offset_tzyx_a_minus_b": [
     -39,
     10,
     53,
     -227
    ]
   },
   {
    "a": "44b6_12dfb391",
    "b": "44b6_3a861e03",
    "shared_nodes": 52,
    "shared_divisions": 0,
    "offset_tzyx_a_minus_b": [
     -86,
     6,
     -114,
     75
    ]
   },
   {
    "a": "44b6_12dfb391",
    "b": "44b6_7e557709",
    "shared_nodes": 99,
    "shared_divisions": 0,
    "offset_tzyx_a_minus_b": [
     -15,
     -34,
     227,
     -152
    ]
   },
   {
    "a": "44b6_12dfb391",
    "b": "44b6_9bfa6a0a",
    "shared_nodes": 62,
    "shared_divisions": 0,
    "offset_tzyx_a_minus_b": [
     74,
     -26,
     132,
     -89
    ]
   },
   {
    "a": "44b6_12dfb391",
    "b": "44b6_b2c44266",
    "shared_nodes": 12,
    "shared_divisions": 0,
    "offset_tzyx_a_minus_b": [
     88,
     -4,
     -87,
     87
    ]
   },
   {
    "a": "44b6_12dfb391",
    "b": "44b6_d29c9ab2",
    "shared_nodes": 16,
    "shared_divisions": 0,
    "offset_tzyx_a_minus_b": [
     -98,
     -18,
     104,
     -83
    ]
   },
   {
    "a": "44b6_144b256d",
    "b": "44b6_66f9292d",
    "shared_nodes": 18,
    "shared_divisions": 0,
    "offset_tzyx_a_minus_b": [
     -91,
     12,
     -78,
     -9
    ]
   },
   {
    "a": "44b6_1574802b",
    "b": "44b6_abf82518",
    "shared_nodes": 37,
    "shared_divisions": 0,
    "offset_tzyx_a_minus_b": [
     61,
     -44,
     -19,
     -143
    ]
   },
   {
    "a": "44b6_1574802b",
    "b": "44b6_deabac95",
    "shared_nodes": 51,
    "shared_divisions": 0,
    "offset_tzyx_a_minus_b": [
     -47,
     38,
     -45,
     188
    ]
   },
   {
    "a": "44b6_1d530831",
    "b": "44b6_24264f12",
    "shared_nodes": 10,
    "shared_divisions": 0,
    "offset_tzyx_a_minus_b": [
     -8,
     -34,
     -208,
     -155
    ]
   },
   {
    "a": "44b6_1d530831",
    "b": "44b6_7a302da0",
    "shared_nodes": 9,
    "shared_divisions": 0,
    "offset_tzyx_a_minus_b": [
     97,
     13,
     98,
     -113
    ]
   },
   {
    "a": "44b6_1d530831",
    "b": "44b6_c96cfa10",
    "shared_nodes": 94,
    "shared_divisions": 0,
    "offset_tzyx_a_minus_b": [
     7,
     23,
     184,
     -180
    ]
   },
   {
    "a": "44b6_24264f12",
    "b": "44b6_8f5ab931",
    "shared_nodes": 28,
    "shared_divisions": 0,
    "offset_tzyx_a_minus_b": [
     58,
     -42,
     -160,
     206
    ]
   },
   {
    "a": "44b6_24264f12",
    "b": "44b6_d78e09d9",
    "shared_nodes": 73,
    "shared_divisions": 0,
    "offset_tzyx_a_minus_b": [
     -2,
     -58,
     -130,
     -3
    ]
   },
   {
    "a": "44b6_2a2eff9f",
    "b": "44b6_cf2536e8",
    "shared_nodes": 8,
    "shared_divisions": 0,
    "offset_tzyx_a_minus_b": [
     92,
     15,
     -62,
     1
    ]
   },
   {
    "a": "44b6_2a2eff9f",
    "b": "44b6_e31261b4",
    "shared_nodes": 11,
    "shared_divisions": 0,
    "offset_tzyx_a_minus_b": [
     -32,
     -55,
     -78,
     71
    ]
   },
   {
    "a": "44b6_2a2eff9f",
    "b": "44b6_eb2880fc",
    "shared_nodes": 54,
    "shared_divisions": 1,
    "offset_tzyx_a_minus_b": [
     -61,
     51,
     3,
     -102
    ]
   },
   {
    "a": "44b6_2f31fc2f",
    "b": "44b6_87bba6c4",
    "shared_nodes": 11,
    "shared_divisions": 0,
    "offset_tzyx_a_minus_b": [
     -24,
     60,
     47,
     -20
    ]
   },
   {
    "a": "44b6_33b596bf",
    "b": "44b6_f28707c6",
    "shared_nodes": 9,
    "shared_divisions": 0,
    "offset_tzyx_a_minus_b": [
     -43,
     -14,
     209,
     120
    ]
   },
   {
    "a": "44b6_341df25f",
    "b": "44b6_81c256f0",
    "shared_nodes": 11,
    "shared_divisions": 0,
    "offset_tzyx_a_minus_b": [
     94,
     4,
     -178,
     -70
    ]
   },
   {
    "a": "44b6_341df25f",
    "b": "44b6_db3c847b",
    "shared_nodes": 26,
    "shared_divisions": 0,
    "offset_tzyx_a_minus_b": [
     2,
     -24,
     -235,
     -47
    ]
   },
   {
    "a": "44b6_3a861e03",
    "b": "44b6_c8e2a523",
    "shared_nodes": 77,
    "shared_divisions": 0,
    "offset_tzyx_a_minus_b": [
     36,
     -16,
     -177,
     -177
    ]
   },
   {
    "a": "44b6_3a861e03",
    "b": "44b6_c96cfa10",
    "shared_nodes": 90,
    "shared_divisions": 0,
    "offset_tzyx_a_minus_b": [
     63,
     12,
     -200,
     80
    ]
   },
   {
    "a": "44b6_3a861e03",
    "b": "44b6_d29c9ab2",
    "shared_nodes": 134,
    "shared_divisions": 0,
    "offset_tzyx_a_minus_b": [
     -12,
     -24,
     218,
     -158
    ]
   },
   {
    "a": "44b6_3bb3690f",
    "b": "44b6_8cc6506c",
    "shared_nodes": 36,
    "shared_divisions": 0,
    "offset_tzyx_a_minus_b": [
     -91,
     28,
     -77,
     79
    ]
   },
   {
    "a": "44b6_40c45f5a",
    "b": "44b6_90724892",
    "shared_nodes": 33,
    "shared_divisions": 0,
    "offset_tzyx_a_minus_b": [
     -67,
     3,
     185,
     123
    ]
   },
   {
    "a": "44b6_40c45f5a",
    "b": "44b6_d5e7d891",
    "shared_nodes": 21,
    "shared_divisions": 0,
    "offset_tzyx_a_minus_b": [
     20,
     -35,
     -214,
     -110
    ]
   },
   {
    "a": "44b6_415c0a3a",
    "b": "44b6_66f9292d",
    "shared_nodes": 14,
    "shared_divisions": 0,
    "offset_tzyx_a_minus_b": [
     92,
     -20,
     -25,
     27
    ]
   },
   {
    "a": "44b6_415c0a3a",
    "b": "44b6_ddf577ad",
    "shared_nodes": 12,
    "shared_divisions": 0,
    "offset_tzyx_a_minus_b": [
     -6,
     58,
     -101,
     -40
    ]
   },
   {
    "a": "44b6_415c0a3a",
    "b": "44b6_eb2880fc",
    "shared_nodes": 10,
    "shared_divisions": 0,
    "offset_tzyx_a_minus_b": [
     -58,
     -52,
     72,
     -29
    ]
   },
   {
    "a": "44b6_53f95252",
    "b": "44b6_97725144",
    "shared_nodes": 35,
    "shared_divisions": 0,
    "offset_tzyx_a_minus_b": [
     -79,
     27,
     -51,
     128
    ]
   },
   {
    "a": "44b6_551a5dba",
    "b": "44b6_7a302da0",
    "shared_nodes": 31,
    "shared_divisions": 0,
    "offset_tzyx_a_minus_b": [
     -91,
     -17,
     42,
     29
    ]
   },
   {
    "a": "44b6_5740d24b",
    "b": "44b6_74d0c52e",
    "shared_nodes": 9,
    "shared_divisions": 0,
    "offset_tzyx_a_minus_b": [
     91,
     -30,
     -40,
     75
    ]
   },
   {
    "a": "44b6_5f15d135",
    "b": "44b6_97725144",
    "shared_nodes": 15,
    "shared_divisions": 0,
    "offset_tzyx_a_minus_b": [
     0,
     -39,
     214,
     -150
    ]
   },
   {
    "a": "44b6_5f15d135",
    "b": "44b6_e57ff5c6",
    "shared_nodes": 57,
    "shared_divisions": 0,
    "offset_tzyx_a_minus_b": [
     -65,
     50,
     -47,
     67
    ]
   },
   {
    "a": "44b6_668e0cc7",
    "b": "44b6_eb2880fc",
    "shared_nodes": 62,
    "shared_divisions": 0,
    "offset_tzyx_a_minus_b": [
     50,
     -53,
     90,
     46
    ]
   },
   {
    "a": "44b6_71a4179f",
    "b": "44b6_9bfa6a0a",
    "shared_nodes": 18,
    "shared_divisions": 0,
    "offset_tzyx_a_minus_b": [
     -91,
     -11,
     57,
     -46
    ]
   },
   {
    "a": "44b6_7a302da0",
    "b": "44b6_c96cfa10",
    "shared_nodes": 44,
    "shared_divisions": 0,
    "offset_tzyx_a_minus_b": [
     -90,
     10,
     86,
     -67
    ]
   },
   {
    "a": "44b6_7e557709",
    "b": "44b6_9bfa6a0a",
    "shared_nodes": 22,
    "shared_divisions": 0,
    "offset_tzyx_a_minus_b": [
     89,
     8,
     -95,
     63
    ]
   },
   {
    "a": "44b6_7e557709",
    "b": "44b6_d29c9ab2",
    "shared_nodes": 74,
    "shared_divisions": 0,
    "offset_tzyx_a_minus_b": [
     -83,
     16,
     -123,
     69
    ]
   },
   {
    "a": "44b6_808952d6",
    "b": "44b6_81c256f0",
    "shared_nodes": 37,
    "shared_divisions": 0,
    "offset_tzyx_a_minus_b": [
     -72,
     4,
     175,
     103
    ]
   },
   {
    "a": "44b6_808952d6",
    "b": "44b6_aaf8b0ea",
    "shared_nodes": 12,
    "shared_divisions": 0,
    "offset_tzyx_a_minus_b": [
     88,
     -16,
     -157,
     -36
    ]
   },
   {
    "a": "44b6_8cc6506c",
    "b": "44b6_d2f34f90",
    "shared_nodes": 114,
    "shared_divisions": 0,
    "offset_tzyx_a_minus_b": [
     -71,
     39,
     -104,
     76
    ]
   },
   {
    "a": "44b6_8f5ab931",
    "b": "44b6_949adeb1",
    "shared_nodes": 118,
    "shared_divisions": 0,
    "offset_tzyx_a_minus_b": [
     -61,
     35,
     170,
     52
    ]
   },
   {
    "a": "44b6_8f5ab931",
    "b": "44b6_c50204e0",
    "shared_nodes": 4,
    "shared_divisions": 0,
    "offset_tzyx_a_minus_b": [
     17,
     13,
     50,
     254
    ]
   },
   {
    "a": "44b6_8f5ab931",
    "b": "44b6_d5e7d891",
    "shared_nodes": 102,
    "shared_divisions": 0,
    "offset_tzyx_a_minus_b": [
     35,
     56,
     101,
     56
    ]
   },
   {
    "a": "44b6_8f5ab931",
    "b": "44b6_d78e09d9",
    "shared_nodes": 57,
    "shared_divisions": 0,
    "offset_tzyx_a_minus_b": [
     -60,
     -16,
     30,
     -209
    ]
   },
   {
    "a": "44b6_8f9ecab4",
    "b": "44b6_ddf577ad",
    "shared_nodes": 28,
    "shared_divisions": 0,
    "offset_tzyx_a_minus_b": [
     -93,
     -6,
     45,
     -31
    ]
   },
   {
    "a": "44b6_90724892",
    "b": "44b6_c15fded2",
    "shared_nodes": 7,
    "shared_divisions": 0,
    "offset_tzyx_a_minus_b": [
     -39,
     2,
     239,
     50
    ]
   },
   {
    "a": "44b6_949adeb1",
    "b": "44b6_c50204e0",
    "shared_nodes": 28,
    "shared_divisions": 0,
    "offset_tzyx_a_minus_b": [
     78,
     -22,
     -120,
     202
    ]
   },
   {
    "a": "44b6_949adeb1",
    "b": "44b6_d5e7d891",
    "shared_nodes": 28,
    "shared_divisions": 0,
    "offset_tzyx_a_minus_b": [
     96,
     21,
     -69,
     4
    ]
   },
   {
    "a": "44b6_95029e92",
    "b": "44b6_d754aa59",
    "shared_nodes": 17,
    "shared_divisions": 0,
    "offset_tzyx_a_minus_b": [
     -64,
     19,
     -218,
     78
    ]
   },
   {
    "a": "44b6_996155de",
    "b": "44b6_aaf8b0ea",
    "shared_nodes": 30,
    "shared_divisions": 0,
    "offset_tzyx_a_minus_b": [
     -90,
     22,
     -8,
     -115
    ]
   },
   {
    "a": "44b6_a21120c2",
    "b": "44b6_d78e09d9",
    "shared_nodes": 22,
    "shared_divisions": 0,
    "offset_tzyx_a_minus_b": [
     50,
     9,
     -54,
     222
    ]
   },
   {
    "a": "44b6_a2bb48bb",
    "b": "44b6_c50204e0",
    "shared_nodes": 4,
    "shared_divisions": 0,
    "offset_tzyx_a_minus_b": [
     -99,
     -25,
     23,
     72
    ]
   },
   {
    "a": "44b6_a2bb48bb",
    "b": "44b6_d5e7d891",
    "shared_nodes": 78,
    "shared_divisions": 0,
    "offset_tzyx_a_minus_b": [
     -81,
     18,
     74,
     -126
    ]
   },
   {
    "a": "44b6_c50204e0",
    "b": "44b6_d5e7d891",
    "shared_nodes": 164,
    "shared_divisions": 1,
    "offset_tzyx_a_minus_b": [
     18,
     43,
     51,
     -198
    ]
   },
   {
    "a": "44b6_cf8fed6b",
    "b": "44b6_eb2880fc",
    "shared_nodes": 29,
    "shared_divisions": 0,
    "offset_tzyx_a_minus_b": [
     87,
     9,
     90,
     -96
    ]
   }
  ],
  "data_dir": "biohub-cell-tracking-during-development/train"
 }
}


In [ ]:
import hashlib
EXPECTED_SHA256 = {
 "tools/divnet_infer.py": "9ad85ab0418870e06b40a2ab2108d18dd9e5e6507b01a4c6429e7fc716783d72",
 "tools/daughternet.py": "c609eb7ca3044b87a59917c4b7e925e6cf7940728f4f44a71a21a1fd11272e26",
 "kdn_core.py": "bc5e02f8ec2eb172a962eb3e2fa9401c57b6bf01c3534f6bd593d0348af25c7a",
 "splits.json": "0fd3cff10858b4387a3d03bb2dc80b0ad8206a76b4bd8cf580bec75a4c8ff213"
}
_bad = {}
for _rel, _want in EXPECTED_SHA256.items():
    _got = hashlib.sha256((KDN_SRC / _rel).read_text(encoding="utf-8").rstrip().encode("utf-8")).hexdigest()
    if _got != _want:
        _bad[_rel] = _got
if _bad:
    raise RuntimeError(f"embedded source drift: {_bad}")
print("embedded sources OK:", ", ".join(EXPECTED_SHA256))


In [ ]:
# ---- configuration (every knob is an env var read by kdn_core.config_from_env) ----
os.environ.setdefault("KDN_SMOKE", "0")        # 1 = 5 division-rich videos, 1 epoch, 2 folds
os.environ.setdefault("KDN_MAX_VIDEOS", "0")   # 0 = all videos
os.environ.setdefault("KDN_EPOCHS", "1" if os.environ["KDN_SMOKE"] != "0" else "30")
os.environ.setdefault("KDN_PATIENCE", "6")
os.environ.setdefault("KDN_BATCH", "64")
os.environ.setdefault("KDN_LR", "1e-3")
os.environ.setdefault("KDN_EPOCH_SAMPLES", "0")  # 0 = one pass worth of training samples per epoch
os.environ.setdefault("KDN_CONT_PER_VIDEO", "60")
os.environ.setdefault("KDN_HARDNEG_PER_VIDEO", "40")
os.environ.setdefault("KDN_INNER_VAL", "0")    # 1 = early-stop on fold k+1 and train on the other K-2 folds
os.environ.setdefault("KDN_CACHE_DTYPE", "float32")
os.environ.setdefault("KDN_DIVNET", "1")        # in-sample DivNet comparison (giorgosi/biohub-divnet-v2)

if str(KDN_SRC) not in sys.path:
    sys.path.insert(0, str(KDN_SRC))
import kdn_core
cfg = kdn_core.config_from_env()
if Path("/kaggle/working").exists() and not torch.cuda.is_available() and not cfg["smoke"]:
    raise RuntimeError("K-DN full run needs the GPU accelerator (2x T4)")
for _k, _v in cfg.items():
    print(f"  {_k:20s} {_v}")


In [ ]:
state = kdn_core.step_prepare(cfg)


In [ ]:
state = kdn_core.step_build_cache(cfg, state)


In [ ]:
state = kdn_core.step_train(cfg, state)


In [ ]:
state = kdn_core.step_oof(cfg, state)


In [ ]:
state = kdn_core.step_verify(cfg, state)
import json
print(json.dumps(state["timing"], indent=1))
for _p in sorted(Path(state["work_dir"]).rglob("*")):
    if _p.is_file() and "kdn_src" not in _p.parts:
        print(f"{_p.stat().st_size / 2**20:9.2f} MiB  {_p}")
if not state["verify"]["pass"]:
    raise RuntimeError("contract check failed: see contract_check.json")
